# Test domain samples — VOV và Nhà thuốc Long Châu, 20 URL/domain

Chạy các ô từ trên xuống (**Run All**). Kaggle cần bật **Internet**; chỉ dùng CPU.
Notebook nhúng sẵn **40 URL**: 20 URL của `vov.vn` và 20 URL của
`nhathuoclongchau.com.vn`. Không chọn `vov2.vov.vn` hay `tiemchunglongchau.com.vn`.
Mẫu reservoir ngẫu nhiên không trùng URL, seed 42, giữ nguyên ID/URL từ
[`links_corpus_vi_en_zh.parquet`](https://huggingface.co/datasets/AIGuruTinix/ViBioMIR/blob/71be0e1239a19dfd7a717e2bc13e3bec21bd0334/links_corpus_vi_en_zh.parquet) (1,220,598 dòng).
Nguồn có 9,784 URL VOV và 79,598 URL Nhà thuốc Long Châu.
Không cần clone repo, tải Parquet hay upload code Python.

Dùng snapshot crawler và probe hiện tại: tải trực tiếp URL mẫu, retry,
giới hạn tốc độ, tải HTML/PDF, trích xuất và kiểm tra chất lượng (QC).
Hai domain chạy song song; từng domain xử lý tuần tự 20 URL.
Kết quả sẽ khác theo thời điểm và môi trường mạng.
HTTP User-Agent là Mozilla/Chrome; Accept và Accept-Language được đặt theo trình duyệt.
User-Agent không đảm bảo hết 403; kết quả thực tế được giữ nguyên.

- **VAO_DUOC**: lấy được HTTP 2xx và không gặp challenge; xem thêm trạng thái trích xuất/QC.
- **KHONG_VAO_DUOC**: đã chạy nhưng không lấy được trang đích dùng được.
- **CHUA_CHAY**: chưa có checkpoint, không tính là lỗi.

`robots_http_status` là HTTP của robots.txt; `http_status` là HTTP của trang.
Chế độ probe tải trực tiếp trang để kiểm tra truy cập, không dùng robots để chặn bước này.
Requests và Selenium được ghi riêng; HTTP 200 vẫn cần kiểm tra nội dung.
20 mẫu mỗi domain chưa đại diện cho toàn bộ URL trên website.


Phạm vi cố định: **`vov.vn`: 20 URL; `nhathuoclongchau.com.vn`: 20 URL**.

`NETWORK_CHECK_DOMAINS` chỉ gồm hai hostname này. Mặc định `MAX_NEW = None`
để chạy đủ 40 URL, log tiến độ mỗi 5 URL.
`BROWSER_TEST_MODE = 'failed_only'` thử Selenium cho URL Requests lỗi,
trang JS, text rỗng hoặc QC lỗi. Có thể đặt `off` để chỉ thử Requests.


## 1. Cấu hình

Mặc định chạy đủ **40 URL (20/domain)** đã nhúng, với `WORKERS = 2`
và `MAX_NEW = None`. Nếu giới hạn `MAX_NEW`, lần chạy đó chưa kiểm tra đủ mẫu.
Checkpoint đã có, kể cả lỗi, được bỏ qua khi chạy lại ô crawl.
Muốn thử lại toàn bộ URL, chạy lại ô cấu hình để tạo thư mục mới.
`RESUME_RUN_DIR` chỉ tiếp tục run có cùng danh sách 40 URL và snapshot code;
run 97 hostname cũ cần tạo run mới (`RESUME_RUN_DIR = None`).
`RECOVERY_RUN_DIR` có thể trỏ tới run cũ đã giải nén để tái sử dụng raw hợp lệ
cho ID/URL trùng mẫu, đồng thời tải các URL còn lại trong run mới.


In [ ]:
TARGET_DOMAINS = ('vov.vn', 'nhathuoclongchau.com.vn')
SAMPLES_PER_DOMAIN = 20
SAMPLE_SEED = 42

USER_AGENT = 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/130.0.0.0 Safari/537.36'

WORKERS = 2
DELAY = 1.0
LOG_EVERY = 5            # Tiến độ mỗi 5 URL; URL lỗi luôn in ngay
ATTEMPTS = 3
ROBOTS_ATTEMPTS = 3
ROBOTS_CONNECT_TIMEOUT = 15.0
ROBOTS_READ_TIMEOUT = 30.0
ROBOTS_RETRY_BUDGET = 90.0
ROBOTS_FAILURE_TTL = 30.0
NETWORK_CHECK_DOMAINS = TARGET_DOMAINS
BROWSER_TEST_MODE = 'failed_only'  # off / first_each_domain_and_failures / all
BROWSER_USER_AGENT = None  # None: dùng User-Agent thực của Chrome đang chạy
BROWSER_LOAD_TIMEOUT = 45.0
BROWSER_ATTEMPTS = 2
CHROME_BINARY = None       # None: Selenium Manager tìm hoặc tải Chrome phù hợp
PAGE_CONNECT_TIMEOUT = 20.0
PAGE_READ_TIMEOUT = 45.0
PAGE_REQUEST_BUDGET = 180.0
PAGE_ATTEMPTS = 3
MAX_NEW = None           # None: toàn bộ; 10: thêm tối đa 10 URL
RESUME_RUN_DIR = None    # Hoặc đường dẫn thư mục run đã có
RECOVERY_RUN_DIR = None  # Run cũ: tạo run mới, reuse raw 2xx và thử lại các URL lỗi

from pathlib import Path
from datetime import datetime, timezone
import json
import sys
import uuid

if sys.version_info < (3, 11):
    raise RuntimeError('Cần Python 3.11 trở lên.')
if type(WORKERS) is not int or WORKERS < 1:
    raise ValueError('WORKERS phải là số nguyên dương.')
if type(ATTEMPTS) is not int or ATTEMPTS < 1:
    raise ValueError('ATTEMPTS phải là số nguyên dương.')
if isinstance(DELAY, bool) or not isinstance(DELAY, (int, float)) or not 0 <= DELAY < float('inf'):
    raise ValueError('DELAY phải hữu hạn và không âm.')
if type(LOG_EVERY) is not int or LOG_EVERY < 1:
    raise ValueError('LOG_EVERY phải là số nguyên dương.')
if MAX_NEW is not None and (type(MAX_NEW) is not int or MAX_NEW < 1):
    raise ValueError('MAX_NEW phải là None hoặc số nguyên dương.')
if RECOVERY_RUN_DIR and RESUME_RUN_DIR:
    raise ValueError('Recovery tạo run mới; không đặt đồng thời RESUME_RUN_DIR.')
OUTPUT_BASE = (Path('/kaggle/working') if sys.platform != 'win32' and Path('/kaggle/working').is_dir()
               else Path('/content') if sys.platform != 'win32' and Path('/content').is_dir()
               else Path.cwd() / 'runs')
RUN_DIR = (Path(RESUME_RUN_DIR).expanduser().resolve() if RESUME_RUN_DIR else
           OUTPUT_BASE / ('domain_probe_' + datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')
                          + '_' + uuid.uuid4().hex[:6]))
if RESUME_RUN_DIR and not (RUN_DIR / 'manifest.json').is_file():
    raise ValueError('RESUME_RUN_DIR phải chứa manifest.json của notebook này.')
RUN_DIR.mkdir(parents=True, exist_ok=True)
print('Output:', RUN_DIR)

DIRECT_PUBLIC_PROBE = True


## 2. Cài dependencies

Cài các package crawler cần trong môi trường hiện tại. Phiên bản thực tế
và log pip được lưu trong output. Khi resume, dùng phiên bản đã lưu.


In [ ]:
import subprocess
import importlib.metadata

packages = ['pyarrow', 'requests', 'trafilatura', 'pymupdf', 'tld', 'ipython']
if DIRECT_PUBLIC_PROBE and BROWSER_TEST_MODE != 'off':
    packages.append('selenium')
requirement_file = RUN_DIR / 'notebook_requirements.txt'
install_args = ['-r', str(requirement_file)] if requirement_file.exists() else packages
installation = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--quiet', *install_args],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    encoding='utf-8', errors='replace',
)
(RUN_DIR / 'pip_install.log').write_text(installation.stdout or '', encoding='utf-8')
if installation.returncode:
    print(installation.stdout)
    raise RuntimeError('Cài dependencies thất bại; xem pip_install.log.')
versions = {name: importlib.metadata.version(name) for name in packages}
if not requirement_file.exists():
    requirement_file.write_text(''.join(f'{p}=={v}\n' for p, v in versions.items()), encoding='utf-8')
print(json.dumps(versions, indent=2))


## 3. Danh sách mẫu và snapshot crawler

40 mẫu giữ nguyên ID/URL nguồn: 20 VOV và 20 Nhà thuốc Long Châu.
Mỗi domain có `sample_index` từ 0 đến 19. Snapshot được kiểm tra SHA-256
trước khi import; notebook kiểm tra số mẫu từng domain trước khi crawl.
Manifest ghi file nguồn, revision, hash, seed và phạm vi mẫu để đối chiếu lại.


In [ ]:
SAMPLES = [{'id': 704563,
  'url': 'https://vov.vn/suc-khoe/4-loai-rau-ban-nen-tranh-trong-mua-mua-post973112.vov',
  'domain': 'vov.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 706114,
  'url': 'https://vov.vn/suc-khoe/bo-gan-tam-binh-to-chuc-gay-quy-tang-benh-nhan-gan-ngay-viem-gan-the-gioi-post1035616.vov',
  'domain': 'vov.vn',
  'sample_role': 'original',
  'sample_index': 1},
 {'id': 706377,
  'url': 'https://vov.vn/suc-khoe/cach-cho-con-bu-khi-me-duong-tinh-voi-covid-19-879521.vov',
  'domain': 'vov.vn',
  'sample_role': 'original',
  'sample_index': 2},
 {'id': 706678,
  'url': 'https://vov.vn/suc-khoe/cay-thuoc-quy-khong-trong-cung-moc-tua-tua-khap-vuon-it-nguoi-biet-tan-dung-post1244137.vov',
  'domain': 'vov.vn',
  'sample_role': 'original',
  'sample_index': 3},
 {'id': 706928,
  'url': 'https://vov.vn/suc-khoe/co-nen-an-bi-lon-thuong-xuyen-post1102200.vov',
  'domain': 'vov.vn',
  'sample_role': 'original',
  'sample_index': 4},
 {'id': 707097,
  'url': 'https://vov.vn/suc-khoe/cuu-song-be-trai-nguoi-nhat-bi-chan-thuong-so-nao-khi-dang-trong-thoi-gian-cach-ly-880653.vov',
  'domain': 'vov.vn',
  'sample_role': 'original',
  'sample_index': 5},
 {'id': 707299,
  'url': 'https://vov.vn/suc-khoe/dau-mua-khi-lay-lan-the-nao-post946723.vov',
  'domain': 'vov.vn',
  'sample_role': 'original',
  'sample_index': 6},
 {'id': 707479,
  'url': 'https://vov.vn/suc-khoe/dieu-gi-xay-ra-khi-ban-ngung-uong-nuoc-ngot-post1010466.vov',
  'domain': 'vov.vn',
  'sample_role': 'original',
  'sample_index': 7},
 {'id': 708399,
  'url': 'https://vov.vn/suc-khoe/loai-cay-kho-ngot-nhu-duong-la-vi-thuoc-quy-viet-nam-co-nhieu-lai-it-duoc-chu-y-post1263715.vov',
  'domain': 'vov.vn',
  'sample_role': 'original',
  'sample_index': 8},
 {'id': 708669,
  'url': 'https://vov.vn/suc-khoe/loai-qua-sieu-giau-tinh-bot-an-vao-lam-cham-lao-hoa-lai-ngua-du-benh-post1213223.vov',
  'domain': 'vov.vn',
  'sample_role': 'original',
  'sample_index': 9},
 {'id': 708775,
  'url': 'https://vov.vn/suc-khoe/loai-rau-viet-ra-cho-la-thay-la-than-duoc-bo-xuong-ngua-ung-thu-post1129703.vov',
  'domain': 'vov.vn',
  'sample_role': 'original',
  'sample_index': 10},
 {'id': 709340,
  'url': 'https://vov.vn/suc-khoe/nam-dau-bep-suy-da-tang-sau-3-ngay-che-bien-thit-lon-post1036522.vov',
  'domain': 'vov.vn',
  'sample_role': 'original',
  'sample_index': 11},
 {'id': 709402,
  'url': 'https://vov.vn/suc-khoe/nau-nuong-thuc-pham-khong-dung-cach-lam-tang-muc-cholesterol-khi-ban-su-dung-post978352.vov',
  'domain': 'vov.vn',
  'sample_role': 'original',
  'sample_index': 12},
 {'id': 709448,
  'url': 'https://vov.vn/suc-khoe/neu-ban-bi-nhip-tim-nhanh-hay-thu-nhung-loai-do-uong-nay-post972732.vov',
  'domain': 'vov.vn',
  'sample_role': 'original',
  'sample_index': 13},
 {'id': 709673,
  'url': 'https://vov.vn/suc-khoe/nguoi-viet-chi-tien-ty-moi-nam-mua-tao-do-an-kieu-nay-thanh-dau-doc-co-the-post1155173.vov',
  'domain': 'vov.vn',
  'sample_role': 'original',
  'sample_index': 14},
 {'id': 709963,
  'url': 'https://vov.vn/suc-khoe/nhung-lam-tuong-ve-vaccine-covid-19-khien-nhieu-nguoi-khong-tiem-chung-875456.vov',
  'domain': 'vov.vn',
  'sample_role': 'original',
  'sample_index': 15},
 {'id': 710876,
  'url': 'https://vov.vn/suc-khoe/su-dung-hieu-qua-thuoc-nham-trung-dich-egfr-tki-the-he-3-cho-benh-ung-thu-phoi-post1012057.vov',
  'domain': 'vov.vn',
  'sample_role': 'original',
  'sample_index': 16},
 {'id': 711104,
  'url': 'https://vov.vn/suc-khoe/tam-dem-bang-nuoc-lanh-lien-tuc-tai-trai-nguoi-dan-ong-mat-thinh-luc-hoan-toan-post1206514.vov',
  'domain': 'vov.vn',
  'sample_role': 'original',
  'sample_index': 17},
 {'id': 712207,
  'url': 'https://vov.vn/suc-khoe/viet-nam-co-nu-hoang-cua-cac-loai-trai-cay-moi-ngay-mot-qua-bo-ngang-to-yen-post1191038.vov',
  'domain': 'vov.vn',
  'sample_role': 'original',
  'sample_index': 18},
 {'id': 713042,
  'url': 'https://vov.vn/xa-hoi/lan-dau-tien-cac-tram-y-te-tai-tphcm-co-300-loai-thuoc-post1126708.vov',
  'domain': 'vov.vn',
  'sample_role': 'original',
  'sample_index': 19},
 {'id': 213425,
  'url': 'https://nhathuoclongchau.com.vn/bai-viet/bi-di-ung-thoi-tiet-nen-boi-thuoc-gi-nhanh-khoi-53300.html',
  'domain': 'nhathuoclongchau.com.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 214251,
  'url': 'https://nhathuoclongchau.com.vn/bai-viet/bi-trat-khop-lung-phai-lam-sao-cach-nhan-biet-trat-khop-lung.html',
  'domain': 'nhathuoclongchau.com.vn',
  'sample_role': 'original',
  'sample_index': 1},
 {'id': 217630,
  'url': 'https://nhathuoclongchau.com.vn/bai-viet/cac-trieu-chung-khi-mac-lao-phoi-la-gi-va-benh-lao-phoi-co-nguy-hiem-khong-55060.html',
  'domain': 'nhathuoclongchau.com.vn',
  'sample_role': 'original',
  'sample_index': 2},
 {'id': 219040,
  'url': 'https://nhathuoclongchau.com.vn/bai-viet/cach-dung-chi-nha-khoa-co-can-chuan-nhat-59725.html',
  'domain': 'nhathuoclongchau.com.vn',
  'sample_role': 'original',
  'sample_index': 3},
 {'id': 219815,
  'url': 'https://nhathuoclongchau.com.vn/bai-viet/cach-lam-toi-den-bang-noi-com-dien-don-gian-68993.html',
  'domain': 'nhathuoclongchau.com.vn',
  'sample_role': 'original',
  'sample_index': 4},
 {'id': 220312,
  'url': 'https://nhathuoclongchau.com.vn/bai-viet/cach-phan-biet-tri-va-ung-thu-hau-mon.html',
  'domain': 'nhathuoclongchau.com.vn',
  'sample_role': 'original',
  'sample_index': 5},
 {'id': 225132,
  'url': 'https://nhathuoclongchau.com.vn/bai-viet/chup-mach-mau-so-hoa-xoa-nen-dsa-la-gi-dung-trong-chan-doan-benh-nao.html',
  'domain': 'nhathuoclongchau.com.vn',
  'sample_role': 'original',
  'sample_index': 6},
 {'id': 229499,
  'url': 'https://nhathuoclongchau.com.vn/bai-viet/dau-tuc-tinh-hoan-nhung-khong-sung-co-nguy-hiem-khong-67806.html',
  'domain': 'nhathuoclongchau.com.vn',
  'sample_role': 'original',
  'sample_index': 7},
 {'id': 230885,
  'url': 'https://nhathuoclongchau.com.vn/bai-viet/dieu-tri-viem-va-cap-tinh-o-tre-va-bien-chung-neu-khong-chua-tri-kip-thoi.html',
  'domain': 'nhathuoclongchau.com.vn',
  'sample_role': 'original',
  'sample_index': 8},
 {'id': 238312,
  'url': 'https://nhathuoclongchau.com.vn/bai-viet/hop-thuoc-chong-soc-gom-nhung-gi.html',
  'domain': 'nhathuoclongchau.com.vn',
  'sample_role': 'original',
  'sample_index': 9},
 {'id': 246897,
  'url': 'https://nhathuoclongchau.com.vn/bai-viet/moi-mo-xong-nen-an-gi-kieng-an-gi-dam-bao-suc-khoe-70824.html',
  'domain': 'nhathuoclongchau.com.vn',
  'sample_role': 'original',
  'sample_index': 10},
 {'id': 249470,
  'url': 'https://nhathuoclongchau.com.vn/bai-viet/nghien-ruou-nguyen-nhan-trieu-chung-va-cach-xu-tri-51141.html',
  'domain': 'nhathuoclongchau.com.vn',
  'sample_role': 'original',
  'sample_index': 11},
 {'id': 250706,
  'url': 'https://nhathuoclongchau.com.vn/bai-viet/nguoi-tre-van-co-nguy-co-cao-bi-gai-dot-song-co-42025.html',
  'domain': 'nhathuoclongchau.com.vn',
  'sample_role': 'original',
  'sample_index': 12},
 {'id': 254687,
  'url': 'https://nhathuoclongchau.com.vn/bai-viet/nhung-hau-qua-cua-benh-suy-dinh-duong-o-tre-duoi-5-tuoi-41829.html',
  'domain': 'nhathuoclongchau.com.vn',
  'sample_role': 'original',
  'sample_index': 13},
 {'id': 264708,
  'url': 'https://nhathuoclongchau.com.vn/bai-viet/tac-hai-cua-thuoc-nhuom-toc-la-gi-nhung-dieu-ban-can-luu-y-de-nhuom-toc-an-toan-66186.html',
  'domain': 'nhathuoclongchau.com.vn',
  'sample_role': 'original',
  'sample_index': 14},
 {'id': 265237,
  'url': 'https://nhathuoclongchau.com.vn/bai-viet/tai-sao-nhan-vien-van-phong-thuong-gap-van-de-dau-co-vai-gay.html',
  'domain': 'nhathuoclongchau.com.vn',
  'sample_role': 'original',
  'sample_index': 15},
 {'id': 266033,
  'url': 'https://nhathuoclongchau.com.vn/bai-viet/tap-yoga-nen-tap-luc-nao-la-tot-nhat-57374.html',
  'domain': 'nhathuoclongchau.com.vn',
  'sample_role': 'original',
  'sample_index': 16},
 {'id': 268911,
  'url': 'https://nhathuoclongchau.com.vn/bai-viet/thuoc-kalimate-la-thuoc-gi-nhung-dieu-can-biet-khi-su-dung-thuoc-kalimate-69001.html',
  'domain': 'nhathuoclongchau.com.vn',
  'sample_role': 'original',
  'sample_index': 17},
 {'id': 281452,
  'url': 'https://nhathuoclongchau.com.vn/bai-viet/xet-nghiem-hpv-o-nam-gioi-va-nhung-thong-tin-can-biet-56801.html',
  'domain': 'nhathuoclongchau.com.vn',
  'sample_role': 'original',
  'sample_index': 18},
 {'id': 282916,
  'url': 'https://nhathuoclongchau.com.vn/benh/mnemophobia.html',
  'domain': 'nhathuoclongchau.com.vn',
  'sample_role': 'original',
  'sample_index': 19}]

SOURCE_PROFILE = {'rows': 1220598,
 'id_min': 583,
 'id_max': 7466161,
 'duplicate_ids': 0,
 'hosts': 94,
 'host_counts': {'article.iiyi.com': 729,
                 'ask.39.net': 32772,
                 'ask.familydoctor.com.cn': 1,
                 'baby.39.net': 3368,
                 'bachmai.gov.vn': 2783,
                 'baidianfeng.familydoctor.com.cn': 60,
                 'baoangiang.com.vn': 10738,
                 'baocantho.com.vn': 12016,
                 'baochinhphu.vn': 2302,
                 'baodanang.vn': 14500,
                 'baogialai.com.vn': 3290,
                 'baohaiphong.vn': 16286,
                 'baolangson.vn': 1981,
                 'baonghean.vn': 15941,
                 'baophutho.vn': 1288,
                 'baoquangninh.vn': 2912,
                 'baoquangtri.vn': 5483,
                 'baotayninh.vn': 1198,
                 'baothanhhoa.vn': 782,
                 'baovinhlong.com.vn': 5113,
                 'benhviennhitrunguong.gov.vn': 1200,
                 'benhvienvietduc.org': 425,
                 'bingli.iiyi.com': 5,
                 'byby.39.net': 37,
                 'cancer.39.net': 736,
                 'care.39.net': 1282,
                 'dantri.com.vn': 2063,
                 'ek.39.net': 448,
                 'fitness.39.net': 185,
                 'fk.39.net': 928,
                 'food.39.net': 594,
                 'gan.39.net': 294,
                 'gc.39.net': 59,
                 'giadinhonline.vn': 16729,
                 'gk.39.net': 90,
                 'hanoimoi.vn': 1283,
                 'health.people.com.cn': 50,
                 'heart.39.net': 1055,
                 'hellobacsi.com': 11747,
                 'jb39.com': 527,
                 'jbk.39.net': 2033,
                 'khoahocphothong.vn': 8143,
                 'laodong.vn': 31282,
                 'man.39.net': 382,
                 'medlatec.vn': 24762,
                 'mega.vietnamplus.vn': 2,
                 'nhandan.vn': 5225,
                 'nhathuoclongchau.com.vn': 79598,
                 'nk.39.net': 762,
                 'pf.39.net': 85,
                 'phunusuckhoe.giadinhonline.vn': 27577,
                 'phuyen.baodaklak.vn': 6252,
                 'pmc-ecm-healthblog.beta.pharmacity.io': 5333,
                 'pubmed.ncbi.nlm.nih.gov': 240000,
                 'qihuangzhishu.com': 29,
                 'qy.familydoctor.com.cn': 2,
                 'shen.39.net': 325,
                 'sj.39.net': 197,
                 'special.vietnamplus.vn': 1,
                 'suckhoecongdongonline.vn': 154503,
                 'suckhoedoisong.vn': 85823,
                 'suckhoeviet.org.vn': 5627,
                 'tamanhhospital.vn': 4261,
                 'test.pmphai.com': 2989,
                 'thanhnien.vn': 72111,
                 'thaythuocvietnam.vn': 2927,
                 'tiemchunglongchau.com.vn': 13281,
                 'tienphong.vn': 3021,
                 'tnb.39.net': 717,
                 'tuoitre.vn': 3893,
                 'vietnamnet.vn': 15543,
                 'vnexpress.net': 2055,
                 'vov.vn': 9784,
                 'vov2.vov.vn': 6899,
                 'wei.39.net': 225,
                 'woman.39.net': 687,
                 'www.120ask.com': 98160,
                 'www.a-hospital.com': 4701,
                 'www.baidu.com': 11,
                 'www.cnkang.com': 40046,
                 'www.familydoctor.cn': 2,
                 'www.familydoctor.com.cn': 26819,
                 'www.msdmanuals.cn': 905,
                 'www.pharmacity.vn': 69,
                 'www.qdnd.vn': 3700,
                 'www.sggp.org.vn': 1853,
                 'www.vietnamplus.vn': 3695,
                 'www.vinmec.com': 26129,
                 'www.wujue.com': 708,
                 'www.youlai.cn': 4139,
                 'www.zydcd.com': 204,
                 'youmed.vn': 7189,
                 'zhongyibaodian.net': 6061,
                 'zysjonline.com': 6591},
 'first_1000_host_counts': {'bachmai.gov.vn': 999, 'special.vietnamplus.vn': 1},
 'sample_host_counts': {'article.iiyi.com': 0,
                        'ask.39.net': 0,
                        'ask.familydoctor.com.cn': 0,
                        'baby.39.net': 0,
                        'bachmai.gov.vn': 0,
                        'baidianfeng.familydoctor.com.cn': 0,
                        'baoangiang.com.vn': 0,
                        'baocantho.com.vn': 0,
                        'baochinhphu.vn': 0,
                        'baodanang.vn': 0,
                        'baogialai.com.vn': 0,
                        'baohaiphong.vn': 0,
                        'baolangson.vn': 0,
                        'baonghean.vn': 0,
                        'baophutho.vn': 0,
                        'baoquangninh.vn': 0,
                        'baoquangtri.vn': 0,
                        'baotayninh.vn': 0,
                        'baothanhhoa.vn': 0,
                        'baovinhlong.com.vn': 0,
                        'benhviennhitrunguong.gov.vn': 0,
                        'benhvienvietduc.org': 0,
                        'bingli.iiyi.com': 0,
                        'byby.39.net': 0,
                        'cancer.39.net': 0,
                        'care.39.net': 0,
                        'dantri.com.vn': 0,
                        'ek.39.net': 0,
                        'fitness.39.net': 0,
                        'fk.39.net': 0,
                        'food.39.net': 0,
                        'gan.39.net': 0,
                        'gc.39.net': 0,
                        'giadinhonline.vn': 0,
                        'gk.39.net': 0,
                        'hanoimoi.vn': 0,
                        'health.people.com.cn': 0,
                        'heart.39.net': 0,
                        'hellobacsi.com': 0,
                        'jb39.com': 0,
                        'jbk.39.net': 0,
                        'khoahocphothong.vn': 0,
                        'laodong.vn': 0,
                        'man.39.net': 0,
                        'medlatec.vn': 0,
                        'mega.vietnamplus.vn': 0,
                        'nhandan.vn': 0,
                        'nhathuoclongchau.com.vn': 20,
                        'nk.39.net': 0,
                        'pf.39.net': 0,
                        'phunusuckhoe.giadinhonline.vn': 0,
                        'phuyen.baodaklak.vn': 0,
                        'pmc-ecm-healthblog.beta.pharmacity.io': 0,
                        'pubmed.ncbi.nlm.nih.gov': 0,
                        'qihuangzhishu.com': 0,
                        'qy.familydoctor.com.cn': 0,
                        'shen.39.net': 0,
                        'sj.39.net': 0,
                        'special.vietnamplus.vn': 0,
                        'suckhoecongdongonline.vn': 0,
                        'suckhoedoisong.vn': 0,
                        'suckhoeviet.org.vn': 0,
                        'tamanhhospital.vn': 0,
                        'test.pmphai.com': 0,
                        'thanhnien.vn': 0,
                        'thaythuocvietnam.vn': 0,
                        'tiemchunglongchau.com.vn': 0,
                        'tienphong.vn': 0,
                        'tnb.39.net': 0,
                        'tuoitre.vn': 0,
                        'vietnamnet.vn': 0,
                        'vnexpress.net': 0,
                        'vov.vn': 20,
                        'vov2.vov.vn': 0,
                        'wei.39.net': 0,
                        'woman.39.net': 0,
                        'www.120ask.com': 0,
                        'www.a-hospital.com': 0,
                        'www.baidu.com': 0,
                        'www.cnkang.com': 0,
                        'www.familydoctor.cn': 0,
                        'www.familydoctor.com.cn': 0,
                        'www.msdmanuals.cn': 0,
                        'www.pharmacity.vn': 0,
                        'www.qdnd.vn': 0,
                        'www.sggp.org.vn': 0,
                        'www.vietnamplus.vn': 0,
                        'www.vinmec.com': 0,
                        'www.wujue.com': 0,
                        'www.youlai.cn': 0,
                        'www.zydcd.com': 0,
                        'youmed.vn': 0,
                        'zhongyibaodian.net': 0,
                        'zysjonline.com': 0}}

BASELINE_DOMAINS = {'vov.vn': {'error_type': 'not_tested',
            'requested_additional': 0,
            'selected_additional': 0,
            'shortfall': 0},
 'nhathuoclongchau.com.vn': {'error_type': 'not_tested',
                             'requested_additional': 0,
                             'selected_additional': 0,
                             'shortfall': 0}}

EXPECTED_URL_COUNT = 40
EXPECTED_DOMAIN_COUNT = 2
BASELINE_ARCHIVE = ''
BASELINE_ARCHIVE_SHA256 = ''
SOURCE_SHA256 = '87e44a792d7a99fb0fde5dff7f09f1a38a0a1fdf02ddd1ac93a005736e00b0a7'
SOURCE_FILE = 'links_corpus_vi_en_zh.parquet'
SOURCE_REVISION = '71be0e1239a19dfd7a717e2bc13e3bec21bd0334'
SOURCE_URL = 'https://huggingface.co/datasets/AIGuruTinix/ViBioMIR/blob/71be0e1239a19dfd7a717e2bc13e3bec21bd0334/links_corpus_vi_en_zh.parquet'
CRAWLER_SHA256 = '3e09f6489a4e639b9bf9c44f0500aba96118a0d96751b53cf15433b2de4183cd'
CRAWLER_PAYLOAD = 'c-qZ<ZIdKNdEoc{6{S_^s(aPb(=+>GX;v**tpsdnwUSpWILB;L)O1%*m!`X`RaHGNnw<}rzy=Ooz<~+4v%?ZP!VWM9+Z<=K-0@)*`X3v?O5)S~g?rvIE3+!Ar)Lp&md$K+Rpyg#Pd@p)=ecv|z8F6jCof&U<qgvCV6v9XCP&^po+Nq4yP0@zETZ(tn}j>jBx`%qWJv$!X&BGq*{+wQUOXE{hw$7vckWzclqORz2u6#1kw!t_#nX9`=3Y3PC3%>~$t-I$)Vp+d9;R8ubq2{~5)G(?s&p+`%<?F0d&6iHE++Xf9^|U<Y_LevXqI=lC0*zCIE}*LD@ihW;V>F3@&u}fSujYZ^GTFPLtQ_Q4)aO8qw2|<X*dgaqm)Yx!#v94X{5@i-!Odm`(`qeu1C`_o^%#@Jki~tfnoU|PXbj>4Se^_c&?tuVK#<tsGs}~yy#4$JcNp&{xT1TLl|hLUc8<qGxa<T^D*~j4o}9gUcnFYoQAVuGS$Cycd{{dUH?i3dr_``;i6@EJkZT%NBS|RxrUY3FN{GhX7K=!g@!;qTci^l7tM_79=v-!iDzwm$mWwcw~D68PLfmkR8_czuRa$~qF3m(#-;gDn5M}AEH%tP>178Of00M@aZvsU@Bzhh`uIjMIq)vai;<?`D4v9XX)>66GE`mKjq+eL88#YUeBsus*I&Nr^*n$7vfp^^Dn7lGyctg>;rgXc*K2+Wwn}o4c{gu+8{JO#iU%()U%cWSUcT7!uFe5NzZC7f80YJkF8o>N!ey`d;umhebffJ};=RcGd^FfgTHduWv>&Z+TmS(0f8ML%C`@Ci;x}%+{M^g8gR7r^0U9aV_Zy7{>>_WG51Os5h6n$p5e#Rhk-9TW4w?$D0htG_PMjs9Bn8}Rwxs$n-i2vt;-cD61dTh{INZE+nG1Br(V<jO^<fgD0LwNJPX)w5oR0}n`DG`WN3*7%?)WWSlo1zlv_-(Afl=G@xozLoZkQ+2co6Iy<xxhhv^`@C_yXpt=z@NW*nvc5x;Ko|CjZEKx6=jC0HAqU9wdA8ldI&>bPj;1CXOnYh0};YcJRr2(i{1mW4&!Uh&u5QU^ZJPoqRg?3oxC7GzRLyJ*K`EpVDYP2?vq%l}E_MktWFalAYIOc#gqixz2Dgoo7ub2V<FKNUp<d5XZgGg_8`%hh%-0_cmLdXofqv=`Zrp+Eac@H91MbVbP+E57Yn_$ePB=;<y7OvrPgCQ_w2|#flT@C2Y^Eehj|fLE6zY#*(^5(t>IC@63D;aM{Dtis9CcYMcPx00FYc(^LeCK6S7Vttf!>0pljgL6mB2o8hLJ#BWAT9IF7rN5Bn(EOv@={>{%_^V?iTK&nOd@1)TAHVwggtKON=)2NdzcABZbeJA(sEV|vW+kX-ZdIG&=4oa84GdsS}K4E~mF`nJSPL`+fyxF4ZJH+WjI31o0>OiZd%E<WV&`a1VO*r7y`X!58?J#`Xf#2Mfc;q$l<GSa=*&B~w*LHA4VXU;lGDWVa-(d`A#}~uNB6<OcciQw{UgSX9y>No4b%dnB3q1ws3XE`phYd~YV4NiIBlKp`fw#5>jdo=6vSg7CBCCJ=N(jLS_w*Z`SNNpB)2C^Ehat>Ioop}$J|N&j1#u(u9_xAAemuna1g!D<*b88TfIf8cqj{78fem6{JAmP6mLZJdXfkYK?G_bYItwox&VhA`;7DR<UOWVjYv|#!0FMTZBYU-&&N4ORc$o2^cuR&mdQ5Fqv`2mKxA-45QmELOEhZD-sDL@c5zPl3dInMqjKXY?H=#_c15DTa2+#&5VFL6V-z^)$OL3M7KRH|gZVljAQVVnadEIYn3A--5VUu5v;uL6c&Bw>R*KXbL!bLt#fJh#Bapq-<(I_6oK;H<h=V=0m2{eDj+lLK_N1``^K^}QK5qyUqQ9c+W#pc$a%1#Jxa5uoh0p(2tdI7|#m_1r^W6)R)mjWtL;SLvVc437Rn5;>RCst8jW*jjlhPeS&Yz8|YAhdB|TMC<C7f%r!?Bg$w^L(CtW@~+Y`^)RJ@e4v`{#Tj)k$-(gBUe)nGq&X&L+!`X6PbGTAMlXC`y(k!CyETLLsJ)8+naq&3C$Y3F56tO?>*V`Hp+v{7SpD=h@E}x9BlrU_YCk#-EMUdX7MU)fXN#f$gB4(!LLAu2`tJ$Gt2N(&rgVggojlWPCo9v@WvvXtl^;mlxUI+NYG)vCXH}gr18#z-XRWPxs1P#fE3|r#CY|_0%RnZSd?=*$Y$<EM;UC(6t=eXzSZ`+-0HzN#(u!KsG2oldQ>*@$;!e1Q<Vy)(OW&0&y?4VD9WnsssC)4dU;aV>OGN2I=AQ_A-qvEREHEW`1=X0GEPY$0XA`;2;q*RS(O$n+nufQj0}JQYpZPua#?f9)@<ARcEkEu+Gc0pB{-P+Ju(N0mJWC|DXOSdY7$RlJZP<1FE2BaV+4mQM-{58hMzm(92oIo^BC#>mYymiYg-<XA)i~IulR6cj|emc4vZjm0S*PoDIx&;Zx;?9WAv?Rosm?l?Xl|hHkIoG{Im3%v5B{K&yyY3O!;uYnMbZsPrb(df~BPSY<sX_dy{awGYq|S%S+`1W$1aB=192{6cK>)5A!@FoQM9z!+^d6g4_i03m~})By;s$vrbeCPc(Xshi#KuP;Zuzv{!*?h(~b7B|wSR*5ctBL&JS8KLJuRIvh+ES-c;$H1ja>bh~&0VM|3rSu|m0q>jIUqZYO#oRCdnj-KuLZH5Q~S1+IOd9qr4Ynv6#lvnHc{Q1;eT4Ll=bqt>rYo$84e7%g#!cEca)=)avf%-ggV=3SYv8Bfbq!S|gC_)O~wzfuR(6I)8(lJh5buJw9*tWcGiJUW`KCvrw7IS1|nu<EY=wxpC#$0fRT8c?uRnlcKO&E6uLTvj4SB+uvDOHH2sxV<ez$_P6f;z{#?><fqEFVyAr2+$na~klB;YOI!@&ell4;HT}pTk4>93I-Ai#9Zi38VoV>q1pJmf;!KNguZKR+tR<#t#M{W<rob3G9+4chXi<R4PNu#K2DKrpp$Jtce=H-U8$a!W`)uDs~Y4VZO2m-}qOre}0iJZiAloa9!zmac6<5a}@7li9wRi7a6{z6AE9EV%IlD$m<24fQykW44XPjCi@W_dTqU+qJ7yibUY#?nZR%wfRDzOV5!sw@tq`v&JWLzkh$WS(<IpJprV}sO$ZfrmZ+_wn}H@`^evfVzk$<$Uz+QnS4<fM3j~qN{dbPlwH1unS%%am0>C_!C_jm?LAsbJ+Fwu}khAyWWRY2tt1jO%^k(o1>G5{o;(kzJs%d~t(2@z!Z`Hs$_5-j5HVZtD&K*NlCYC2O)^jeQ1z<R2_(CX{fwZ`YY&uGfK+A=0381vf(770Q0I9O>4YKs=Ba7VXEB#tQ67p}6%HaC*>nJo%<BVjhj$hLURN;ZHGCT-SDTNwKI6ODIUGJHmRRRSwMAMwMTNhcc3ofAsAldV-Mx&ePSL*B}p!w*LRiQJCP}z-Uf+p19)0dTIEhkqcdQt>I=bPD_V0quoY}L&ijsSN=?c6bUJb)g99w`k6+<IF#O_mr?C!p5D?-%b!@eJqypXSwh<GGe255S*R1c?eL1E^@V>}V411}M5gC14_`X0;U17+72w2ygxsmWpefQLAhT31q7Q31ohRv<DZtEl^JzF;js6<;ot?a)o3TH3!XInhXGa=v?X1sZs@8J*EumfZHDRUuxxC{lcCo`NG`HMOjx9NZ2app{B5lnH*Bif%!6gKJWR5(A+sF$#PSTd|Y=B)G-OrLuniJnkVpT1gFhd*09kSOu{VluAzCA_sR>0gJ_Nl0U?a2H4WeZi}N4=QD`zEei(=a+CFozF7`krY_Zs&E~Em;0MkS9aWO~PvZLWK1SE}Ivgpz0P^}fEwS9omU<W4QQE#IIJQAEsV1EEyz0mf~qkbMlhb+)<@px)iiG>t{eLaIo4ba8}fO{L45Ttx$F+skdX{#O%D6i9_U}rG|obNr|HGqs@n5g{E^9jKy&lnAdo@h++th%pGQCvb9bunTaP-);$Q$OO#Q2Hve7_&2_kEW=B1h46W-%?yr%o<l`A1wWqn?RSVOc$WA+Ci20Jjvo5xFe-NHfgrm8ncchT$g|${yVzWcAg!@f4&<zaEm%WtP0P>rvjMyc6CSvx|xuQx2r-j(6az!ygad(GZ0PN9w{0`W-ToO>b0hsLtkv@im5titw`A;>f-9>CM-HqtR&1^x)^#CfPG?w^gv3|H|`N|H{F%g%EvH_6OCvLjd&!5adns`s|2Xltic4=qPk&Oi)DQ=W*;?DuHw~_b{o5kkT}-Ds(S-GD4M07fKqrbleTBT_rM0Rdtw&gSGdbg+Q$E!V*?dJoE}lxN{ivd>KDC1Bb}o5*k%B@Ce%KSc2QM*L~0QOuSio}jI~5%lqr$xf+UrTuC_@ST%H!A`3%?$I0C74qdActfQKY8s2B_hXjH{7VSQ1iBPB>x%03*OSEZfSDRsbaARK~XN-<2w{%as^u3g;)_6#=cYgeHIR|j}-lb{T%xIWIOlQ!@%q)Y%1>+|90${W3I=jrzO_4D*}$v<h8Js7HO^Q?T9+-(LRB94=xo+o9JYTCveb}?cNb77h`r;$srT``ga#A8_mWqvXi`K`h#Nu^EGw!8vfDJ{L41N(50RFHuPfWhbZF`G?@ZFlvJJZMzUh=1^&f&=~H#S5*ofQHdPWh84u+>Hp;Xzp?HtvT}YM47jc0!91i*ru3#@O8r%Z0Yl8ZPtM?+(%O&aH(^!h38_XQ4|^pFwti6m_j)ZJ+$K4C~21FvTY3)sy1PO)fMXX)S@u2@Km=|V-8flV4Zj#$YT?`vwg9@-R-02POv*kcEU*=kv1oCG0VbHq~Q4NGA0;XR`yho;PJs!gB0qdlvk`GFXMUkl0$xuUkqV)f$k^jm9!YN87v$xKH+K6Qyx}q`b0t;WFY&@^2rf%*q#{G?8->pkULi7S5rDoi~+iZld6Y`W3wj@YjwiG8w<b}gO)@SSrqHL8anDAyT#bOHJX&pG=UlkxPkydJQdMlRpx-);Oit%5Z!O_N_c!sf!H_>;gAJ-a1g?|3f474LySE$wgZr9yGg79GbMdB&nQw};5?De!qNFMcA8VD^?113sEucP%VuH~Ts_-d>>Q~n7yt_du6MJNE`qv{O`>St#157|mgh;Eaqw6zb>1Q5@>VGr8TEww|EV@;iQF0)#|+kJV9@WBoN%par5-vfkk(4oakFrr(#$(lJ~eX!iukCBihN;4iw>@Pn`~gyOmL~XLocB@6~$Y$2!n}ms4ZE7Le3OXI2HX1+V2mv>L*h!WMIHi6X2e-!FVx4qLPD3e?qEO*fP$NBl1Op(qdF3plqcU)mQWKrArqsx9V4i8+npFYY0}Wwk!&kRQ%axrqlX$PvBHUixe8=M&M2g-Z1$%oPj+Hp`Cbz#*sE0s2cs{ohfY`g|X1Ti<f9`)Y6bjRfC?i&N{9tKG%1f7gs5)mQ46E3b?HIf3l6}3HPlLt!9);+axS-TPc+_esX~TeHYsWYYkgxF(Ci@&NVPQU5Ao0mg}C6NpUK*1uS|{D7Q_XQ+IC^wU$H4RN2r!h$h|I6%fT3a+DW0pKg1Xx)+vU2U8bgq)G}Pz5Oke1=p^QV0Yp)cb6cE^SD?tD)m!DZ;`R<;YXx1G{o!Wt)1GPrCoByFGv?V_zaUgoi{a4K%bX%-WWi$7(v^+4LUD*d>IH0%)|lxY@ObB6hYj$Y%nm}$h`C&=QTJ5tI^gRRV=Bw_C>>XLj$wK#KkGNx}nk8#bC8vy<D;Zis^QCf*8^iG}*8-ZLuE|CagYCdfqhNjc2B_2!2J$3XL1|5uK>lxl$)T%>9PN5Rn{O%Mxjl*wdw-tX0qyV?s#`^!4Nr$4>tKF?Z%<>sVu5FassniLczbTdnP_jmw4Q1xx|p#!lY7$3E8v>75_6Q;;4myrE{xR&~`EH2KP-TkGd5(g-Dy1@J`?%0_?jG@#+4OAP!b{-RLb%o9*0z$!<$382p`Iw=hzP|O)Jv8FzOAwf}sWC^I!Ht`wcWVO+D>yDrFHmZW=+0H99&q{eh1L*m;Sd~Tnta6Dm&R`FmwtBva1;H{_(i}B%ZB>gXoO*i=SNFC{3(_xi_l>$H?1{AoJ5T0KzZiq%o@HnsWc-Bu;Gm9?Q<2~xFlM6e1AB8DEx+jEfG4)Cp8y^N;(t1}UoD5gDa|D!@aX~!d!Wt>q^M&wp?!jVESp4HD{vZ}C-{kQn)?Zk8gp{Kno~iwhfZpmP^+T1O<!?Y)V%vZ*@<?^wenOM?9_9xj_u)HVk$fI=r_8X7rpcJPy5s}>eT;~P~uR18yXYM<kmJQ12EN|r|@QqC9CH*Z~@eIHvAWQYSK(Yeb3^WuJM;OJl&CP@>?rTmF8^)T<>^q%iCvr2v|c9X;njF{y<8#r~GN*`=c&NQ~4dN<<e>8$L%YMy%s7q5Vt5iwN6$X7GjmSQ}j%ai`Ejh|3KLzh4`<-D^S+1tHY4VM{(?-UygsQPN$O~JeJLQ5~Rnl9Zx#DZ~C?ES@BiMSc*Ok=wCpdCEBlqy)~wYrM+(J6#Oy0A?Wx|$u9*x^1uLeSjRF7tEkQ7D1XDdt<{i~&mL?G>Ht4*soU+h1ttIs$oSnOb1N1AOF61JgP^rs3Uf>Ib?jeh8h~q?qyIq$;_?i95c)*57eMG70x=Od^c7!9HPvuS+fw_<^fb+~cB*u!<Yv=2D;^6N;}tpU$ZGd0tG|l4cA*O#%7yMG{<(mE{tW*-#a>kwg}I?N=Mq7oI36X;xFF&JYrBBcUn2dxJk{!1je5`b8hPI4?NFFkOp_e_?A4F>8x5Td5zM0aoXKR?c$XQGg1BzvEmnj;5}l*)Lceyg{AbYQ<*P72;I+Z^RmU?>?BX+zfln1-RZXd7dqwdB22X`MMH_pl^|L5TDwB&5)gGF_0R_B7d25hN)2fLoK$aXjQHnipYNl&hJ&t#87BQ1}j%T%WPfq|=prPY~1HVdoj_yS;nr(DjTYtJCY&*6{wI8s7FV^Z?YS(wf4wA<V>cl_W)&ZEY=qH7j{=)6suXq|->n!`zVEnCtfyN%RNrd8;N_*{cxBLHcxCJt=K3-O_n1f@<vMSru$Tv>W=FGBCjc~T=X|JSW_^909Y=;;4&^D&5B`)N3;<MXphJZ9RA1$g?-8;}!y~@yAr=w7_<Z?r>HE7L}fRn<jHyG&Y2QeOo&03h{?zUFsh>>)gUdqZU?~>8B*Eag6Qm_)vSJ^E$HCeWrWVSEX9_{5-ZFmjosqTR4esU@kmLE(tN{hxR5HvSjaHuXrf6-k-ozv?%wK!WV?6i_WUGqcBjOtRzUhbU`l97xsFn-fJf8J}VzO8{K3Dijv*a^uFFo6hKl{=f4T6IQe0rp3lo*&7=Y+^SqB<M$BeIi-hYoZ}uSHoo<EHkW>X;s1HpGhBO?<&;~9E?L&z}FSIpw@_878Eo#_4E^nMYdqls})K_b&Ji{OACV#^i;t(&c$$A_I%R%P3f9w01Xcl;z+ny;bCIBJ=J<ViJ}^h*W<bP)SN&pJp7nNIf(PdL94tZmMD=5W3uf%Vms_rROd?b9teUvW>Ok&R3>g%sM~FyPqzW_8HXnzZ5yTDW+%?ssO6$9++vO0f?;kmpnl;-Q&<jU;0-s=SXp7O=K}0HiMcp?Xjz++H0CwYC?&$CCZL9{&9!fnTSOvnP;GdrdVu~hl&ao|)0l0C9msS%byE`6LEeK<-lc57&R;kSL+n5?<n+`GyLmh#dc#mA$$^VOBF#E27MPhN|4V^Ft;VN=V1_M-5(yL#3bMt!qFrp^5@FQuaT;qqN|}lo?WxB`8PQL1{t~uDQ*u&%D26;@F0|}9D^lAaoCTx~Y#J%STp2;?<*GO%cNmhbEK<ux$ewNr>jw1FY>f6=*N8x;Jt}pSb{*eZmCWH#6(j}tpsJKO9n}}yuVN#m(z0%(#55{bjwGq0S4ba4Jb8wfBSx#4jwY6Zsdzi)#1Exl%b>8hJXKl%r-?!hMfrjk$gNICCu1hERI!@%Jdcrvz;d#-kr!jEP8?*u;(hGwjzV<{f5i;jLQWRXYSRgp4{Uy;PVE{+7Ouv&qA7jXCaYK9Rx9&kqiiaxwV^ICo}S%$w#-nby1J@`tk&T5NX9NPbt#J%qh+hHMAK~wk^O3TTG<<91vz`E_Fw7O9a7bWvQ%YYWLcQ1m!j&)B;Y6b!0!c$9^3L9I4@Fi(8EEa&hjT*JBd)zfjWn%8WnU|{l;1<^m*#ypH1?9!ao;TdRD`JW|dJt?%ha$M`ml6c9#KW6np@cpoC8KA*6=|%NWIbve9}m>#V}2J-Ym9j{gj-7&sYTKAYDox<HZs9YiE89FoJGb_~-}AtlXaIaDOU)}L_ct3oAFie(mChiRzrbft6aQ?u&+@s?bNNfu74T{6vvS-u9Th~p9KeG6M{>!c#<eCqZk9rpcbW+j^EBDoS03F~~w6Zpw(psAS&>6Q$`aS|6?2v=G)(*iL+mOGov*HpYZXbbo{4Qw)%G@cVUvS%vACFFm85s+F1*^s3PXLg4|F?NuDhD@~)=RQM`0NjW{rw|V&(HbNcn}$F-#?d0Bgu|3gS}+Ns$qI&wpiy+3Phm{Mc%M)prkJJ;`nySgFVNqM^!F0|y^MczvEdJtb)O(&GM30cDcHY#&_w2-za5$^-{aooaEjR!r#S8mqSW>=eij&~390D-L9ci(V{*X==Y1Gs8=+AFX(ux2GHy<$*{DzVhwXO&D(C!j#oiTh9pSDVf2vZel4uP5x}X^Cu518FJ9C<g{AfZOEFPsG6ywt@F+T6pu0vIxzq|_tcv&Ub?$j+%rtPGDxh^iO+6hH2Q#}zl2$-H_wRE9Y!}hO_F+uZ;fY|kUS*bvURM6WFM$0>5QLJ{0lJ2?9i7a78kfmex?s|T!cJ@x6u0}B;wH3_-uSSAd@w#RpI@vAgHOHXI4Cad+6~ai~zD)>;h>Bf(3vH$7v!-Z`V6UHX@&qA#ZAh((oB~3~nEL_uQ_N9>Id^d9>rE{Ha2sP&ox?e~x~#9C-~Mbo?447TAG}2JbNwRk7zRA3x|FbFqmmvPH4+WmCrZ_T*f`ZvM>my&O*UGiag<w9K2oPNBqnv=lSx*`0JMf-9YiTbOy{YgM7?kr4}okz+RmH1_uoG9KKSkjUz_dT$<A-V|C)D(#~ba<lXc_Yizlt-XCAvVY+1Q{rK<$^)Q*~>K{@|75RN&yaTBO*beZv=7_L7tU4P;)APFs@g`E@p6t8r%bsODeJpw#^s%OZpXbNnkM`7;ilwueNi7+1>oDc^wvR0WSz%(2tA@MWmXz-cHszNqN`9D9>iTvS|!rvw;QV<otw)fJA;$w#tfkZjzqynw&Fv<q79m5$WIjRORuSm!%Cp38|MJ)E`O|DCP34P$0T}%?ql6TA0Y4CmnY{adz#<@R&05X_FYnbqC7p8Dcb{9ElNK;kHD`Z=3bvns?S^~3PN%2kBL_q5VLT@$R5bNWM+t@O&KbR(W2Hk0%^Uk#kGPRb?F}d|;2{tGI;Vb<mu<+b3G1Oq@*{f|We8k3x9$;hJpM}`<QMt`xhB2)|le!?X)$FLLX#jQE-2N&@Ht^?^pM@s(_Vt_IjqCTnb<_Lv*RJ05&)~d>Sjp3ZXi3v<YtFcv_VUm?4O7U6hC~<op!k_R!2FCTJf(_hn%oIAg-Sr$RZxJpU#nxDGg4e7`A-IvYKKEF)n=z7#Z*74bEVklQgZ#z|HD;J){gWP#vX?r#Hq<(uDHoX@U)|ZQ8>W&bYgF8cEfBB-5l}cJ~%k&7;lXIN=gd4rKINQf((Qih)U=Benm5@>)Ok*4CJHX@D3!v-@~e<O%u+t13nTzwRUD1Rt+4D$;qTgS3aFwCzLrINRgGGp@Cp{ducdS0>`wiixbywQEnM|zqYQRddfJjYe)fP&ETP5x1y(yipa_mq=5zd!iI%30_B)}!M03pj?(lqq1jelfRKB$%iy3<70-R@1?Roxnc{BYtg7|mH$<N{#DB(jK$7Hbp!kAh7`#=2H$vR>!9EFLl9x+$F95kWRjISoOIH`Ao{UAs%@E^f3~xzIa!FOKVZv$}6*5e<>)WayAa<S8fh7cR+kaekI9L`sdZXcpJN9;f<cL)=Gnnmx8rdW>#v6moz#!s!bODy*HuTRzm*ogH-k|SR4ZVbIPmd5e2kw#P()5zT(4}F&vJs@ecn;cEWzg*>9QlG`KXeJqO3`{*Fk(if+I^7jACWdm+OsOxpg3l3(M6T&_OP`uQs0s({@C6Q9NpFH>S}Pj@kg~6)=!Tj12{V>MT~=LtUhrX+YA&#R8aU;wxw!(+(s2oN0#$hulO+#^-u9_ydlq_jiERoUbs~cA%|4J0IUP(kz|9j%LlbuyhoN3RQ_<q-eW4>G%*=_Plm?u@E#K4SYO|p*EnEf=)NU6%volz4`q`SH2shN+rK^h`oDhkt)KkitKa)Szx}Vcp7QDkfBo^#-o{t0mdnqw%*i%N?(Mo!8DE7Jb19apSWrHK@t^je>J&A|vGXX?$JpwJm42>2E#oO%ct;1DGF29<o}~jyVjNC#>plLi3F)g!DVhT_wgw>7bk}dm0a;ZP*qLEXlY}Wjz*JKhgMm^z{u8e}|G726{8DBvMsEXHH6lS@4mSle%E<Nwlb3uynIsM24q_@+vX}VS3fU{+!A7q+PBT)of{nvhbO#vW5azLG(--q8_?4~}7>@cbL_OAMh@H}DrD=iqi){b4x158dp0#R>IX0gsQ-C0`mc^Yh7_`!WE$b+l3tA}#Y|c54d1y2)Dz8~2p94$8>)5qQY)eoPC{hp?V0&xRib9ccDh|avt-@AgX>u$O*y)OazzUhN(Nf(6^Gb>~LK2iAxMmWfg_QPh1h-M?%M`&N0LK&?ATUodCg}hx0{k=aKL6b2g^f*KeZZq@M=9Lj2j-B5*p3jC-1HTNDP8*`Z$WL?ga2*cId{kJf2O&8=U}b>WXmt_AE<d<%fI9Z$kn$@50*!0g(r67@`a}^_H|?t-NQzfg<U|t$ZwhLt4rXZApzdm1r6RDZ&uNEz>)3_QBh5v(W^w&C(e3z2PY2O+TGd1e-791!B}0RJt(vBRQE!ck1|9MI05Gi6&WQ)ARGfYd!>?OsR))Z8sDnVt5pH3rbX2y(@P4jXckk>ei5(nMRCtl4M~x4BT4oab60|lN?)}ryEdO0cUX25eJy%ObhSrrP6^$q7O<MP=S!kxy3bHi8BUgT`T;;G`G-g+*!)%8Ey(ZbVi1-38e-ejwG5{BFy{(4E)*0S25Y%!aD*9>0F!%9=0Qat3$h2WECShb(Z-?>ekl`1n;;sKN<<OuD49+lp&_C2oKI4^h0OdUZ_2GG>JcbhK?b2f8+*MGp7lagRju$^LCMwjR+dgl*+!Z}b&V|`lED=Z<idd(RUnq83$FNrIs*+`B}c<ps}r64vwXAG)RpppN^MJtK4)x)*7O_u?gRR!{3xWdneeadDyx9WDEY=(@E4v;Fm8a_)7#Nn6EcgkO{BA%?05EO{;G3~Y%eM_>u0i2?w*C5*s0~#zp5%?8??=Me6!M-<aB1{&;(EWQG0BF-vI0oGT#x`1tyhcf{7ulp&WA7#(-pk?dpsT*-~MzA{7=Apwn#SkOuKTx%I1F{?LQOQ^?@L44;aYF>s!bl16}K9KQW$q%wm8M5u)?O@^30NE{p&dQzID+(M;<2{!6uJH66nrtLyZ=_%Z034YUGb|6#)njX$Rb^t&sQ3U&EeC56`gX$7{daN%fBZsI8TnO4<q2nwj^|X+r(99N*)LRyijAe8QKc5y!5qW^CC>xCH0*ft+0$KN@kXpogZC<xQvb|R8xk`|lkIxvK>2<jYrBcT%trHSU4U-R-;&MwzfWt97Ukkq^yy_u3@JpsIyK>=+isnp8;EXprI4+)X)xi)yYC~q$yf~z|*|#lbL)`fT6}{c|O8f>n%{9~oBU$0ogd{_Y_sr?PPRVOQc=GQoE?n$(eGEAs;2(JL^k_8V3ylXV$X+>qPyqi_9Kz5*Mlu>gfjUFVAcCIq;0PzdJ0QB|Y!jkOGCSz>KOcv3yIldlS`j1JaSWMtM{s7}5iIfU`Ytt9mhwq$ii0k68}70~U>o4IN#!EE=y8UEY0)r#P%jZ4b%01)SLCl($!W<OW42}w?qE8)0<u_~sbq}gsTJ>}WWk*PW3xapi;fEPB}_YNhv4)Z;|0}XXS2W<T@x1L&2fOOu^>SatcB3zZ-4N$`#=8RyZ3*3|5qQr^WiT)_{;l$`N4PYf9wADKYaVc_xj3@!wq;pwg8(H!Yl^NClCmTNwuPzA){(Mg$t&LnBuf<c4@|RMWg3DgBK)-9kqh)Vyw&)gf6&XW@9QAphh`lSyy!%hR9==P~y~@mT5@U+i(^&6lkj8#jfSgho)w{-36~39wVED)@MN%@JNRY;OLc1Zq9fb>%xknYfzObFD7&bst@b4!Q~+#&~%M@5U|`~is(c>>MBk7>f}K9--afG1t|dZ?RGcn0Eq06Ei$~0uz*mAFSfKn=Yf!>t|m=mp-tV3nl_D{!~=U%+ZQF{AV#wXqD|vrivv?Y9zL)T7#;*}qkBldk{xnXC$~S;zhu*7FOtj6+I({#rpFf7Q;1`TC<4^<CE%?i@E~~K|LE7>eE73(Km5(_9)9;*4}bCQ?|=I?((s30|Ml<h{otc-{mm!uee>h*{ME<b|7Uvp?Ry`8|L-6E?LU6}{_h@s{jE=a^_z$P<sUzJ@5lJ<qaXg~kAC>~pS<<n!|#2Sl-&R8yC47TM-Sio?I+*(X_amAfB&0bK78v34}bKJ4}bAP0{!3LdiafBfBY{${Nz`^diYmgfB1{v;M)(rgAQZfZ2aK;pU=U4?7_XSc{}(2X$I-*7x0Q)Td$4p{~TUq_uqlV*m>~&yNNf2Z^k0yI`WX!V$s?auVg-G?_8)`ww(l2peZd#6_3;j^NiQIc&QEord{TO7!npI79!K(VAEi`HkSd>hRy^B3Y$tvqy%+<K)V^t(3?L<2C0pD){b!VWwl`I%2)e&8eJta$HurNRI#=r07$lgR5+y%#GqMxJ{k8*oPTR2;tN+4UFt3;eWr+Ib9H6lVz(%X`!-A}L*z8TFe6|Y-OFmKIbVtgAaEt4{5oDv2_Zf&9L^`8x=im*vt*4trRgc>$7de9vkh&<qr1=0i)ZgX10wsga<q%Kvu?}@If^5x!%eoB!v+WyQgEV)ImU*hbe)O+<>qI$AkoiV$ggu3*;u&1i!<NceI4)D3B758#QrW;nUQK)20gs{hj;$RAO7ygzyG(l9{&7?K%wOKM}Pa%kG}c8AAaMPcXNzw^Y+H~e}$Cw!M*PRC3x`uJMmrh>i$1Xfv7%s|0hQtm2LUexwCpK46)+&WFpu|2#A3CcMj|{rXGBU@jl|CWI=B7;{{yyc9-&ABB6%YaIrVum<ekU&B)#U^>9DrQ-Xj&h|(Si<#+$^({F$B-d8_<`>S^c;T(cELwh-D*Mmm@{|{9raP04@<5b_ZR>%)p1x1{^<e3pkp=4(y9fSimZZ4|6)v#mqSn7D4ub^vKHlLD!vZbT?O0v=xLq?q*Y(Q<OnU&Oxiqm5jZ%Z?XMJ_NzDWS+)CQpGO7bdyhQe?i=f<>84KuFU?{aRe8Dq-Pt8cR^PWG8EZsfZ;ojy4860Hm*i?1Y@KHzmEbtgOC96z_z_GQK3Qu(SG>bM@*u^${JA(_ua8(tM+q)?XqjrAr--ATI77tzL<U`SIws%${`>r}dqLWt|LCvOubeV9M%aJiP{l0rZgR(QuM8Q#<Ocjs$hp)Q+W#B=y~(J=Lm__)k3;IUe~db)Y9Yx1(y<x^-)6IgHei(5`B$qbg=FrW2OY@~3tU>5}V5SGjidaqpH2!(%}MZe$1M3%$nH$3WD{7>HbnM<h~owG{fQJQGo5nE+qhX{>gKsTz){=#Ys}G^Y|uUWvoC$0ossMS>>DP@pq$S`h>F*}j}m!4Ru%OAoC42R(=&ujZCI+47Fz#fdMu2V0FQWW0?|)ZslL2`eYV(HT1H?(JTFU9wFG^c`yW$|Q1hqqESoNwOPoO6XqsifH}G$8g53W@~6#^vCiok_ABP>d5M~kwwANFO1|{U5eYN^))T(-d6i=XV)F)y8EYGu8kT?Rdc&^Gk1U4cwVKVd`U4Jj&KqWRre+FMKV|J*m!^8R%yQJUOTzuu2tOCC~xpkw{Ej9Uz2I4bgy1~$@ikxTx4lqjaqX@C0~l_zIGA;oT!jnKHlWCLIXEHH7WZvt)IFE6`%E;o*tg6=0wSo4Uz54=`uo}ZjDJW*Zvn@xb^Dwmv7?v%JRJ9_Vgg2=YVdo$0pUq?A(6UdC8d1Qr~>Mtu6?QOZyt``#5zGD?7^h^XlxjF57efnIC}^H4BPyW=B~^ekp4c2T`F^OtF((M*Vcb1es9WW{&4uV)p}mW#Qc;4WKuTcBE+*j10DR6atzbaG<MgoEGPu)(Nk>@D;MTn5&YzMQI~8^i_RunyZW4iO()}Fn*(*s1|?O#-o!2B6H*xS=oD6JiInMg!AnH69;&FU}2;e9EB44fpK0C6E;InNwrrEn#;V5>Vn$J2c(=f90w?cI!DgX1wAlzPRhd9rn50l)KNXfg<48&R$dFUSZMRH1!$gfhCnY&#~P4j=qkZt-j^Q~4J$fT)z+&~x&|}cC#!q0JVljiSUeWyw&>z_)iCMy_!}`f`Xl{X;nG*oIq6Yql2CUUyy^&x_Bva1OKUk~g``JDI_a7@IJHZdSzR=yW@oA<I9XPGO?h%4h{d@LZ_z(+!w{gv7;G!x)*v<X0Y;Pw)SjSFIxrI0X>^gN?Jct)qmYEdw>?|li?ZU4G9g_f&1Q=1i|etW{TFZ3RjjW-)Y=P&(O>~vtch)63KLAwK%I~u?7|2o4M}-HcL^Yt<A+YR*qLG=wGzfKRZ&?QGk1av)=r89A+XV5TVPYR(b-}eQA(%~X)o*AhfQf5##PR>gV1{dpu_G!<%?c<Dz^!V6}#KV+tW5mQCRj&V+BUblFZFM1<KsA1QIJ36dwzaDiB##CxF|1hkLfwXWWSfkFK2ZMQ$p9o6-)a;9}RyZk^jU?K54Stq<C7LM(f{km6dQG#CMUG1lJqw?yx8>ReQ}HHBz5iJlJ^(|HETGbs5F3hY3fv{>NNJw2MB8MpjsIeZ1z<w6@^ErQ&e%dZ{ty%Ok(5~mi`&IC?rerl$+jq|z^K%Pkh5Wi4ZP|OD0OAVKbAa_ZQp5nHLH<~T7F*`F?t|!MW?>%A4GjJ27r-%XmB6ne-r7NL<CKLf0d%=%nU*%ufQY6O5^kT!s1TFto182Ym<<HEN*)J0e_*kLVkFtE$?-R#tg7OUbLX<|xp3u1oA}@#e;8ikO?!qJwpCvWMVGe>4FqSEamcV3D_R?mdFoPi4mtW-@b73|2L9okEu?SKUy+h`lW4mHZbxSvjfx|K!A=8lDnc;F}x*P@BG*V2%GZ;K+4RnOAsmG$z0k|Jd9Y`5g{o<q|j+5N%{qV^hcBlttacJ5=(T#;(LHZBKqCnP?l7>jW6Qc-`0p-j4<TK5#ac0QtEQ<D;T}w-4a_?lEfuQIMn<Y;Yan_q=`-9Rotj@%2T#hDy{P(0!m<lVwb6mdPcrm#7+>L85;sNN&_a~{+YBQxVOLPl(0uUvIDY`0ZrUA3%9_98TZwbJVE1(>sgMlyS3_VtfkjL#5dGKm>vnmTEb1@ps@>Oo8!eX!-h?+bPZFzU&<!dhnFZ@TymSX(A`P{$TnWzl!D2^t><+t_Kpxs$e?`-(=+D#`uITOL2vNW&VR0(CU1V&1fwO6#QHQ~`D$`Fl;8Um`Iatg?&rd$Flg)`zRF4jdUVOC-pu_RI+GLL)ssR@(+zw}-QnJvHS%k8O>oqpRXxzhFXb)xIMMp{d)C74&xJ}m^)0zPpehIGnbRu)fhv2p17^XqWq-ZH^7EeoYv&ilIAY(s{W;x6Q-vb5MTE8zlLVj!%ML~AZkUB9J+7@fP8r0HT#;w&hOWS&mWRR+QAJgFksM$g?%m5*Y@$|jvM&2lSED~5$LZq-dGBAJ$^`IEAyBJQOuEh?>vu>uwO%W~xGHw;hUdQhgR!;+)8J_R0elr^bwDM2)b^cG2g2%fHIaU!WHI@z<ufVaz2s(o2y>0~}8amew!+s7r#=onR_$pG$lBQBI8(wgu?+>A;8MXs0JpqW*Xe5r@%!Q*Z8PaO_)NXOv>szpky%54gK9B_G1Bw@5F6T;SUAudG46lDYO3knPp;?^!Y)T)+-eLG(O3dzdrI_ia^3ZM??YrxSf(xuOWXpdQl=76tfsu+Q+As5~`3ec<CrmgG@1bJ<e`C`JRJfyTdy>jw^(Gu#h-5x!kWjReB>8u{rxdldbelGMbXvx-%$njydwOy_ugD7%V3ivHO62BM<#-r*VlxK0mDgtyq@Cs2)#IQ75B7qsZlmkiYcd;=IOCc8i-wbdS3bQI~Ll7kLp>%L&uT4-YS?NI(?~a*tVV@4S&FSnz>ScISU?HF#{MOUjFBlMoY=8q+36!+-iX)Z?%|mTWKF*4*4IMSSMfwgh(K5A@Wau&0m;AN~p<1q$hKBLI@~yDSi`O@osrW;ZhGp~cP!>&j3RKlEZLRCFw}k_yiCWlnF~qioC6stZ5tu3%arrqNo21~W{nCCvtALcErvl|_YKO0co(RYRsjRH9AdO1I2!D;}SFJwE)rRGiRP8RqvOSmDg+jLFLR%*Sz^9-%>iAq!D=tL5T20{sU0e{9uwNqz>^O5bRG_FWA6Vc3ilzl}Zqqs_rI^|*lx+@_9?R)KtB9Dcc7V=p^38;x-eDZh#FdQV)4>=)@0vm(6c<6%Vl|-ir!VoGV4ZEch-MJQlV<1XOF+;lkV9bO-rwl2*a8SXfJ1@ab&8r?DYQ|-4xp(J)m3OXj5Er@OOC5NH&9icPHG%D6t1!u+L0(|p?u00{Z<6vhl@ds&WPM8h$4s}m>z;pBEUZuxgg;m1y^}<&^38tkk772x>U@WPVKYDqh#|_3HAu4mbt9)3(L{4E)1g&wzakfjZC{_0+&PEcUGiiqN*6XEb|Q?|EDr9<!qM#ELMq7AAx~U=1>jBA)a8@F>HC9#CV^9F6v6hjbqom4P+^20<$Ns9c*EbDW5OOnCMucB)Y<9u%ZsOGLf4CHMdvNlfgG-CjusyBHPgCtH-_~>Ldj<K^Am_JP9_{7f{8{NPl2|sO3>`UxU6j05jW8gxgf?>q7y^_!Fh_6Hp3#_IyK-WA2xWwPi4+a4`RI@5Kl2{|=MmLBIiPb|g3c7vtyR<fZGkC=w1-NaxnPYvTv^-kO07YhdPy2cr2E*fRIuCP(C-<no=+_~eQ=xc@IWlMn9w9ehv*KFwKx2I5?43a}-*Bonv@vx?>{`T#Xg@BbXaBn}rwi^UOw1s>e{X9~3V(G1-epxB(lFI4|E{df1^+JpE1VdfnIB23V`+U$r>H0;I<;K$x@0d4F&xc8qxK~gO&2Y<~-Z$G&AKcR_xe{FXX2neb?ARg^qnmn3z4YcN9>u+^7pKv#GbA10_r#NfBk&L@)7Ub}RvFty1|7ReQ#t-hj#|!BV5GSrb@9jRg_uWAStWQwiofVpa$^EyBr0}rv;}3rsf^HVB{6T;8-+6lg`)k+>eEZ<uKLBok$9l(mkv-SvQ0Uzl#>~aY90H<@?-<T20Q$pssLuU&=EV$^lx>V*gbXst5K;59h4P*U^nld@mIc0G+&ZAl06d4Gs|YdyrxrwN5oOP{t7)P9yWW?g9m*JZ7gzFc04OfoUGF7o=dSmKVqahp6Kmo3RW);WahKio*4EZM`n$FKAHNV#O)kQSI>ZgG2<7P9UGMm?Ak-%w|28ET^{&7r^`?*w^eaOlqh1v<54<^NP#Mf$f}Pl=(Ba1)hdF>rr}VJ*--(5|_+R<TSLD<-jD;HAZI-y-CPU#Y?`^iYs^4~I?DD9+Ye0E;Z^7-{AO7;ey?1B3t2JASXm&TNv_VzRZ6ERk-?)DJFM?Yyyz=s`+np($OFD@F2gg=)dNnK#2HT8~3AT0QaN&~=zH#J+QSz7<?YJuMa^-WY-K(4&*%Zamtz6soZmTMZYBfPFeUD2=<K*tZFHV^U$`<Cz-6gV~*4-tfEmHht*)mSnTicYY%^YkqH7y-rGQE~Ybom+i#fjS=TbkI-3r<zK&Q|UrCBK{EYA?P%5v}VI-+29bZHMJI?TR@?78k5E&{2n#1;z-^4o$vG$F2;)D(*@KBH4MgkDt;JJ1scU^;JBCi=v;Fv4jC;^)?fni|HPW1l$i(N^VqG=qc|VSmouS4YNz_8XJ=l12kUb@uY(}C-HkRHiXy~NdTYisZ$Ij_Xd}d9Pha{x<)0FeY|3^U7(!h`>x*xmY;6V;iX$gP|9>f{niU+tdm^5=JMk+CV;X#Rw}hU`$9>jJg2(~Q7a`%^{v8Aq=k&8VZF<Yj0LqQw6XzDxl>c_KF)?AVv#ulF`BhHyOU(6>7SS7uOz>)Eo10rYZ@G6tgMBXP!2L9kom?;w&4nzahStkLNlNpg76(zMvdj>F!ZbJd3vC#4~6xsoJE%c8A?_&C~e!w8g6SFMG%=oU2a2jIE0G%O!|xMmecO4Tb{&_w|qQYy2E?VK^D-k5##lGdk(Lt@`fkU&JZgTU`0tSQ-E%*elg&u76w(#sNxKXsxRgak8&kySrWJ8Fw*|JPk2)|%(ExJag6>^65^cJAy<N&nkycc1d!EZ!l}VzW!LClAr)7n-0TyNdQGQZ!*OA8)w_Y*(fIJS!?u>W;VBP1<nd5MQU@HT0H8~Wq~6tZm-yRP=t~pRM&tPl{9Z3018ktIE<0$tqf(u42vKT_9sZSJB}mnXX%+zs{kD$$HmJ><%u72|ldD0t0~kY?swn3<eG8>KO@D2T$)m4>n|heZFkGigPXb}W0RTjw<h3v`aLL`FYIBfg3`4asm2Qk7#;RG2;j66_(H(%*N=jQ<trFv1LLmKE)yC0e-t(_xX37jB+u%}?EHBt+e<`qJ?tld$jsi<V@#yv!z6-AwVj6)JaSa$aUqQNsZzT8=Z_`>(`2T1zp<|ch=+}e{dH@=I+2pD&lmvB4;DjE?^km>vKO6J}1L{$P)OVl8;!0?Ndwpw{ilf1#4mhBNX01*wQt0?FS8Eq)C!W~J<&{}6%FFURze6Wua36$gKaGY@X+z@v)>LR;%?K~m4^aRiyL1W^jQKki3SMU_5rl3iPoEA&EE{VWYIydpKrqT0$cV<Q*Q2fJXIPKBOrakId8;iltfg)75pxPLm3_U^mN{z&xdS2%r4$_`#|8a^u(PHX#!6Ad#s1u&R4vE@m5Qj%NCmo%0>;i^Aco^0cO+u|TN+!}w`i--Vij0?MecQ<;Xc_7@s(c;tb-%g-l8xT757anO%PDBoj_O`^qtM}A<FA`d}Hh(AZb;YYpDF4%~s?80{7IMWd'

PROBE_KIND = 'vov_longchau_public_pages'
SAMPLING_METHOD = 'targeted_host_reservoir_unique_urls'
PRIORITY_SHA256 = '5532bc0d02218b11b801981a1b9a07c8a8cfaf60a32ae3c807489ece3907e384'
PRIORITY_PAYLOAD = 'c-qZ<O_SS3vf!P+0#6e!0Lmo$L-wqP68%Ec(mX3hBPpb|JsUk82#^GuaDxC1fa-2{^!c6lxVMei*xMfVu%|bNJ<N;uKm1QNGb^)D1r*S1$@AhV9Bu-r%BsrB%F4>Gdi3bghfS2G-gY-nlf`%&UBq6rSj1)NE%MDaDv~nKrrx~BugbXa)^W6qi_%Ln?|uB!E-tHb<mKBc$+IZ+KE!F9CA-b!(W6I$RgrH1Mz+`$MVwWW)vnqVF;v}b^P=)j*9E|Om*?rr>v*xN@?zvgWr&TYaTPCh{VKk$(qu06vR#_;+X25?R5#nWl&6cIldb&z5jrowH&L~g-+8IPZ{eK*`<+pwh~-zg-c?DepLX+Ykt2fin;ZSJN;bM$MFUK-3)R%FNTH<(V6=>-J$UzHo@68Zv5Z$mbRnZo(|CbM)#y6OcQ9ta5tNH|X|+ri)nM@b%l98%y?x`&Jbzmxd685%W9Z#H9)JAGAAI}r`3oojUvCE<{PUl~#BM8p>iLMigLSn@N6~f*6CYt?2iN%Ki|gjwP5Shw+2Q1OqsIr2>DSl($OJfk9c33gpd5t!^Ll*pX7ux#tHhaI<Gbxe0VFhjm6b5-MLZTV!n#LL>TWO?z!<$OuCDUpGK`BNFT%?tTLz0Fx=IlT@%3UjC3wrYijovYSXM=VuP14K6&Jyfz9uU#DU+<MqHGc4=aI)SCIvS;fwwWyZ#nt!;q^;wjbY=zA_k((Jijbc)f4|y0H5Sp2J?9e)RFHh2l$055LT+hCpGdOsI9vNq{SBaD7y)0Ei)QddEwD>;Ahb$hSi#<JA|%=KJxqvD7ah}Nw&(p6;z3r_*VV$s@$V0A7g*$=)y898#shsZE{%aNy!KRa7K5aH~u2cO90K^p)1~1QmtVS*whxr9UC<|5exM~1*vY^012StYF{|2@TbgB_v}~`<7ku7rdGyc4GcTZF5(cmb-*ln<jwQt4Z{{T4J-drR>ai%NE3WHf*&wcE;C80cvA+$JE}-!p&G(En;sn=f9XB;@XxumuoiiKnZ$sKG>?{0{0~uD#)6UnIu}r9yl39mN54A;I$L^kf0-|K8{7$0ZNfJJ)#$)~ST>mZX|A}>M72>DMyjwrZ_{XD05d6%m}v_9OSmZVH~qCLZl+e#RT0N97%Viws=!l#eXF5Z1wX>9&r>Yxd;SDjcz|^$*SMbq{=orOIXFNj8w}5~yb6#&O4#G3({fQH+iFPTY<#J1(s+0dYiEKjZv(s0Ce9Jd+z~|d(ea@!u4v8+zK_yfOe}4rMio;x-{BrqG+CXijniTmn=~oJHUT~m_>0welT^t?R4F##YwqjK05zIUeX9+czh_larSUgE?#jxGyiJaD(LoSYej67+yBnAekM<4N=)8|{k*s)=>Z{9bMO6ijsOA3CE-B&$QCVXj@>(lkSB}edSi*h}RfAPrE!MaN?Er%zEWb>aGl)DdfR9B5@WFM=%+cgqTM%`;t2Mm#w#Y+@k2sodPfy=@3>`KxKqU!J2oXwsO!6JBVs3UK-7r@nto(YpOC4Tg_;-5~3_)xFc_GkWxq<=%?5u*WC~h(Jki}Q9n@Zu4cR5syL+uQN_8Xk<MO;+)H`-Ne{96?;onf4P^qHH0C~{351gR(5U`L@>*pA&BF4pR$s@3j10vTE;nh<kmv@GLPBMkXODrn=;DqMFVm7fGflihX+lM*yA<}AbV*2sJO_}2dpW_bJ@2?QjYnp*B`2r^9!fgkNMj2<Z?*s{wIs&o;BIJ>WiH9eXdg*3DA4QvX?EO3g29%N^zSwzXC!7xWgMOiRcQBs8f6V?&%*IH!L3n%^vSU@p0TBb=BgU}D_a<i3XR6?ULH?oZaF@~#j8Bnb=ALL~Jd_;>YEMX;OReCc+CN-oL%$3%8Sy?sINVEJ3Amdg$+2p`_^DJ5Lq+_q)+a|gWj$!tsPu>^xeh&OP-%NH{f_x}A9C?$khI&|&3|M^_Rm2x@2qW)s=#8;SEs0VaUwD%v@0*z(zA95=n0rmg3jv?7@dKYa$DQT`_{TE8$`0ny;xb>Yh|9;z5_r@mN-|_7EOdDD-SR?Y6oY%$yR_8|O&M_2JX@C3Q~_s=Wq_|o-V@b-!djWe@s{2X%}s$e31OGi0hs(nUMwB6Uq*PmgCpavGI{98V+m-)eTOwPvN*XONx4Nq8O}I)RnBNcBTqFKE|X&BtKX$xv*i!#2$1N}(Uv<xR1fnRp~#m+7a34YvM6a&M(zzDw!nCSNa8sx70r_8@kNq#5s{wL8JGKl2gAt1ehKdrJOIZpVD|*lnF*!sss%D^5g=!0=lB3)gK^W7123@4Lc=(CK1}KQ;U-_=2f`QdW*GQ_cD_{umtG901R<T;36(m41k8{Ql)kH3o`qW<8^HtIHqP+G+YKHt7Q(c{Q6vvhZU~jGf!ko_$QlDY)e)L_6@L!<egJE>#6qN#5Sb#Ze~`7mZpceCHdv==bOSwl0|zDRa|;X877|-Rr~|YR_!&<lEm9f2D!0vHx`5p(%ne%*St1k~JHfvs0CqIqZcoGlYiET`U$qb0@G=HT76|Y5Zs>*?C8doXM6PgJjpC!4sq(iH7ShqM-BG}9a~+zjb=utv2Oa`n&Xb`4K#}IX#Zdu`>j-2V;3o4JdLMhlL460lKPm#gCDNS=Nu#i&1_kf<7rnzG=K8-IbWVvZ@$(5NX<)snovz&lW-VRZ%9;q+PeB*sBH>;dMR|2s%b)59YSJe2pd63K8U;s(e5T|-PDcNbf3!$u=7RZLh`bxk^sXc{F_RyiODt$<0GNheiXvd-z5SpLU!d^BYHsA#lLAGYC1~P>!;y+$CTg&==~st`=Oa&RA3*&PmE{aIrAkj0hN$8R!#!|StKeq$Y2_TWm4IeA?I0kQLtxIfxGA)Skk$ep3tel`q`?vdZ3%7PM9^J44(K{k$9ODhXmS|Fp4!5`k?BU%nB>V>g?0<+6G|e&8=ti~B7Xi{QQM7_`_H?bm3houS1HQs4iW~mLc#S9)%r=+Q9$uOi`o@Q3JvKuJ_$^W$eI2%>_WO)Mx@bb$lbk&5_a5@)H`JO2X05!O7lmg;AXny)jd@DEq>VoUn>P#R1C0o3$n&cPNFhKU3><omgHmn0)<oStlP<O)>X9))ebO|jX>|A3}S`~g<iE|Klg#z(fUiJeIV0MR*jL7;RY>=K>Uq5m6+7jg!^%{s?=jsQ3cOM1lzVlQUO$jsKo=cy(48Fyg1N%?$A5%R*!B)EKOiq@4{Qb?=rc7Qc7QVH@W!vQ46}1jH-MC;`RKdipzl7LA@{lm*z)g#+slt3~*W!%#pW2b~rm6-ZL`-WF|i*U|8GcWpYgi3wT+sqvNlBJ9Bls19p>@sTrp^`SS4Vk@w~C@8}E;gy}ugpPzZZJv<zAtvMTJKgXMPt=_)}7u<speb&mKgEi)I(9PXkON$9VM8cn0YS<HA5~sW*eAG+et{Z~vITNv~XQ6!-z4QWZvIC7E`@LvrjA=FK>&YRU37#At<Nu!E|Naa9?`x$(w=nTqn1N$%Nkan&foWp1Yb}v7kh2=V`LIU-=d*Y%f#3Sbi7hd|%HdJkglA();e-0&U_bTISLzP-B2KMqM|R(5A8O(mC7<KQntcM#s7@0&ZDjt&`y;4=NL4}#Y}%xh^pBI0fP^A85>mb^eT!1GB<17*r(i}pL7_D!GzjP^qD94=QOcRC1myrZ5G(pnP-uyCiSJrUclT!tLiC_tG^x=K#zeAd&h#r&FzH}{I3acU!QPVjN|D$tlU7(;UvQdh`XD__qp<JV2fG9lswq;*%BT6)fZ(Ve_)lvaP2DciqN^(~7k2AlCj8`S?%}Bb7PUVD_orQyLI?5naKxlP^SLz?{RXamdzD5Pp)i0zHDH&tSv!ufp)u;wKd|mCO+Z-T4s_~mT1Q8Fv^*dvF_8_Vc|bfmL>Y2mZpyke+3(s4k?CMBiClCPq0*X!T5IB)^iOL0fotjQH#UC<EOJW=Op~J;<I2BnpL6e~-YvYi^R1EFkYU$M%C%D++c=`mI*+HM`o+U3>oOV46)cB1M1Le(-qV_Ehh#ay;*W&tEuFi1#aj0=#*Xbsle*fap@^%sFhn{G1sf=#%y*y!Miu*gnsYpAgMAjifMk8+6+7jB!M0b>E*)QAqlEBO8$8$X;&KbSD{KKUQ8yl|gtZD?b%lB$GB)C3!k#0DYkH}Q81Kc7B6Vj73im1Gm|Pr?=x-vh(ZKKtdtj1I%5Vi;GwoPv5!Tl60Q7u3LWZ+J!}J`F;XsK=iS{tqzg8g8;qI)_V6`>jHM@z5OLC&X{vCL}U@x?LrpB{Gn{;(UszTLv2+&DHC<F(1v{7>h@6XOkMIL}fR*aw5jyqh=VI=VDsRk(Z!=2z&b$dd?wXJ^9M^YmSDWiQ{QL?IH@epz;!V0wGg5~%hW{m@&3|UTqLl=YWn;Y0bVE0%)MV$l?XBJ=4gedNr7dI1wdGm6i9?3vc7+?^n$*J)GB0`$a1OGA8&3@#<t^{RYKZ+PF9xz>(35bh`Ni^x4oon-}nlxkj>ZbW9T2r?y(oS|UXb5*C&$lO)CRTMwq*xPJ)Aghwa%y8%n<lV5iz0ZNR1z4#9tzh{Z;*nAL?%`TMz-ze!9?=v#ld&)UwfOR1i#P4+N6nQpI9oVJ{S@JvKF&gTds46P0Ws1qPChbPXKgOJ{&4p``4QlJgEt1tvRATxbW!?;54|ZK`u26WSrm(6QNoN%yq(y_MQlUq{tUEv9Q=mSobnHuuRMeAy!KIjvR&@)LlSL&fF(8Ik3Yx+SrY=wLyKFpoe47v@MY7$Q)zsoAyo15MlZdm`7YB3=@g!H0+;02o>w7TpyIl&v6$`HjZs3>9o$vWM#8F{yv|2e&fAUZb=}kgMCy~F=E*R@6P<Gm&1{-xWa*InIMgUxt87%m~u)^2t4n`F^{|!!4XND2?OObGHsql4IMm{7!k8}oaR?_K26dka!o4gKCjuWQVkmoaq4Uqjr4AlBNgCjkp<PEN&<iE3)I6{tC8y8w-~Lnaa5q`5n!k->%jncWlL{5*0r!M)m*cX?Y8PZEILVFlTVWsG<bmdufY$b<M(g_TV=_uK7>0oaE6>h;B;||pg|4H8I)?O!AfdaC=%6^DBR>T@!QW|?o~poX>Y2X1$v+AxxEx8^#7K);a=xdb>s!O;&{3DYG!-Zv;DDoI(MAJngaU0!j*j%Gu-EtaY7DW3`ng{AX|=}X}$(m;1~=qYxLi6uo2z?w)r+7KRYyK4hMf<L&c{kU(B6K>z$Q|W6N*@$H;u;g%wg<OcAbN4@Cbqzct+bl{t?ei+7chyhBvZ&dl@Fpi5oNkv(pQtUfLIpxa7lP?oB{dekJi0~f8eDc4%Ce13VLy{K~$JXjG?pKQ8i4LLcI%bT40^t2?XM&YxOGMV5T7GTg!W6waRK1H2EN2)bTOQMc_N=X7F$X*Mu6<{@3leMPfIVn#LnBY+d#b#AF*?}{`l2ibTN?Ba9JR7f~3O05oJP+NF;rNH@!BT5FM3n}v3|H1kqgFuIh7R&TKps*bK*)}nfxKs1qD2EFKn=sUT5}2@2G77w0F3rRP8}*Vq&|k2U9(On(l@g#Y~8pb8z&sI1RZ8t{S;NRQS#1r02gNs)(T74&3;L@bf;ZMG8(wT3H)bj!2!FvCQn>QGo~t?oj(A{zG%3^Mw9~!TXjnq1j2QE9oW&1076;&CZxh91(QG4U_j109%+By75N1&aZS4KAV{SW^&6adF=TK9x@6#!=?$pjcN<tBC>LzGCJ84(xr6}3%^XFBxTaoQGhqk{(JBnV;X7T8*vk|a8WddaQe32IegQi!u(FwbkVW4-RKe=+Lz!djEFeRFz|fWZ4pMbtt(WtV^P+T$0OJeb2!^zS-JR&(;)Bs;dafbPdWU6$)WNAJSK<rCIzPcA!A_3TLGKRj%8H0%Ty-4G88mt<M7e)SHG7@Kz&YX){r@BqbprOyODI%Nu%eC|nqXu4Xb=G@HJ7{1wgd~O!V)SbP2~C-CGthT=(An<8rM?IKt6{P$`|$24~-E9Rg`zNiSJ$uwco_K1)}+JKTwLNxZu#2;T^7|684rbDi=wjoY>K3pYF=FVHT;U(WWaK&Glrm)4f`&_3pOkR-~8??u^Pp-rg*2eT^othW^1`Ctu9GqxN^f=b8F(-chWH{<9YQHq~ih54wa{3b5VZ15WMIVoU;X_v{M8wof<Wq<u&+SwgD|4_j7b97d0NJrn*uPOU6Wqk><?#Bb1ukSxs)q>9}Krt)Cc^+>oM$)eNF410mX-Q0v_NgX2!87xY(0KR~n0Knb$>}-QF3{Xv)c@Q{R(;cLO>iqox8iK^MTf9&dwm-lYEECK__>Bmq_A36z)}{%&8_AlDU2)c6<{fK9Z#%|Tq-?Lvji`^+wEOu@n_uqH0~J(_vemtoQ?Qt#Qzp7HeKlgg^P`D&DJy4Btf;0zk862wd<Yf4{Ax0kn5Ws%*Z2y8&_b{!%~13kxdWaCUGXAisB&?Xx^~;V3uDUo;K`*BwP<xDT5!vBct|v?wPpeVlJh<`<-3n4*O7LFjD7*5twvS9f$axX+;j>oweZ&z{mMywWI?SRqnH$J$JiO{rm5pa&3T?`l_zEf0fi@0U`0bw{YM(F2k^KNq1NAj4)s?9;NTJIzG{cQ$ViL<1PxzwGe{0j54zL!duOybk9x7j&evsGTc?hie<A1Czo4<RcQAFllp%kEjxSl#QJ9V{>_cY^q>WixFEjuZG4;$l>}&P~6>?w=8_nX2sDU|NIM!j3vI!7{^4BJvN~LF3h*7&UQm*>|g3z!{>5+sQ*6OrBs5b#elpyoq_%6*q<Ogc80((g#e6acq+kOMER?*=eHUZ8DqjZaqW<Zq7b#xKBLWHO*4nlH^2t&ud<#Xx2>hh)r1CoNKGb^yp(SR7Trk;+>^Ua7w6n*|4F^j&9Q)tGDqr?0|*($l{YD)3TE>{XibCTF&s=N+_Goemc#pq6Tz^g=tm33$skbzGWf~N_0V3WVSfpjS3Fmh8)m=d5GWW}g3Qkg%<qxSX?-M6F-QW)dmi|dTI4Or%k+SI=X2xP%Rt9{Gs9mDVlhR3t8uct@1YQLv%Bi%91C7>LY$UWl$t{Ho)PfguOk5k6pt`)W`gLqBIlhU|9(5=EfHz(SqtIw;V?dm9=UzU2fk6RnK)pNnW&Zb=cToOwtRo2W_iUno_G>QUTeP~EEss|U~z_+DT6m>qWfUHlBxo0TGdD@+&z`|{3ngS`sC$l;{K=tVifuFmuhH@&KOMV1bB`R*r;BUOQxZ^EoQ{v8pB_BxQ0>*`EWUf6W2W^&082`ZQV15C2I|^=HC+x<JPTZ6X!m@bPsG&|YgQBk}*6r6y1#tf<s39cVkCqf{_<J=Ml9r0wYeIjr^&loCPFw|8iHn4&CTpP2G<BHhE(=irS014#!Ll^2b{r$!9hg8yj<k6s*~qDc|B{g$6V4cs?3{iN*c*PBKps1AGY)Ec8mYNdxnAg}p5IbibkDo9Ct)?!2-Y)G`32mz%a4P;paTP}K|_RfMvZD*^I6fd^_+F>Pc7DLTiIH^lPw0j7Z}aW&-p+pHea!5YE7E5p=tH~djKDM>R8RXBT3^yl72RhD9IkYH>^{TYRyWvXe;^J&TS5p(}V(POQgm?#3IXrL}!E~BSD``x+vjgWwZiC0_1>$iEx;ubO3p>y4QQIZEi6tJjCL{9k%`V9t8biMzb!jzW?NQGi8`EI{%5bDX|^dm}u(Pq*W=6vm>P3o*Z4BLP(^Jsn#93$f}{S?FOp~I819Q*Cgc833<iZE0hc0vzpC`&`$Z7!c%~T%v=JkN@8^k=zHL}!;PU>1W&aw7}EUI`4u6JLP~kJR!M<*wxY#4WcCg6c_;>J%E*<be*ckPA=Uf+`?o)Qc=<j&efi-u{L|YPFU|8bb_D}Luoo@NvXDGSt394V3&juNJVYF|po|`BjCYJl<neop*IEvy7F1#-r!%8Ct~&PamXBkDdQ9fn%?_Qu!VhDP&d$&@>LfP|pMHbcbs1m#lt%B+?7W^5oM2)pF3Kg(G+p!_YN4|om1!MBaK2n}sOG0wfXSTzONK}00l@~)076}9#G}H`0^@NKV+(wdtEe|Pob4z%F4Q?lsGJTVEtS}W5r-;k6F23ToI+3SX|fmA)hv3$kOQ1L>DaVeoZ%u+pX@a#Rg?kNKlAw$5TWtAH~Hcp;~V~kLAvhDiw){fIovQQnQB4NhkDz{vV(+S7TUz15?HcB`x})<U?La+&%qcD2+F^m)5fXI0NuG%ofm@K>3v}U3MQV@3`jaNYpa`PDVpC`OGV{E6hw_W8*(6!YReO?)#2IUIo7HRYWl)YEjm`DAfzlWEMi8(oGy7bJ~}^>wSGPnZIB`;tq5uedLYMv#2ee&$Nn;dV@H>8P)B@jd!uWFW&WNMcH7~Zt}V27*=zAZ6x+bzDIOEtDHZ^uX*6}F_Z2)e6IYmRNttFKZfBQNCedgP$eqs9D7*AUkZK5*+DbU>V)j||5i>wwW|FI@z<haahs(xD>KdZQs}(E`aOa&kQ4*BqSupU^;&1;uTYJmh%`bobm&&_b|Ly;Rz9aqRU;o#Q_iz90-~Z1qfBl~eFZ<=M|2=^+@MZqD|B*mU^)LVW|Eh_dCRqE3Avzm0NxD!xFoJJn0jplPj>`22$`5)8XBRw5#-UOgQyLTmeM{3c)x35>&u~BoNziypVHp2pKTi_Kds>`AL2n8yULuSQ0G@V&=Gbn@^D)xzQQ&Ok9X7(;1FC0}tDR4lb*hb|uMQVxk~8;8F-twi(kct8ca!hPL2$i;;67c#;w)s5!k3tw?C7h*c9eHx(pdRht4!1^H8duVQKePq^d5O;8pmdm3vyd5<-iDR3Sw4`Tt+%4zk;O$aUFH9;yDvdx!Yo;$u5!VKg9DF^z8@e%3zNiWbT}ZlFU%r^!w5z@;JqC;z;4wu)a)4WFyxPw{x-#FRs+`F|iz&53MT1W=vk?Dgp2=0wFfE@EN6ALWlvlqqi(=G-uA4S*(wB87cHtb|F_kntb94<Wz7wfb`XHZtNR}MHzw6vq`Z-NUaLDECg-Nb?;c+gH&u$B{zjQ@kDa8SQ)5V;oqDzg~!Wj1(e)s?OjrW9t^dWh%qhJx>#<*#bybX4i?k?fJILs06<AeXG$KT7w7zdKa!=jt*Eh?0>nAxt#t_By(CxfWA$EpZph|?MvNT;qgmnbKnSvcBL+xi)oT2;e@|e7Udx{vnD8t5z}u7lD<w<9C=NYhedyN#9vu-G(Jk|>yU8}Ys7K2q!<s|P?Tp*%<`Q&|fqE#}RfCj{VSd^0;sMD=xEOfA8{G(SjT^9s?zPhlOd;+7PX=oCiWsS{8F}iFYZo}VC~|Sq-zP+Sv?Ck(Q*`JN^5Wr?5vXQxi}6-|jCR6Qfv1FB$^nJHdg3~7@0@X)_vVmM6K*Tn(5dG~rA+O!l9*Xy9x1J2=W)bH9P_AUB60fFv$9GsVJqCur=59}8U^Ka=4#`!(u8Ok{69~Q5F>?UcUBJ-65~`#eu{3rPW~--Idto_dJjOI2feP-PRlXr-mrKpt6a18S9{0SL+&|o@8dMn_TtuiI{Cp3jfe_N-3B&E%ET#GUvb!*hml)w#!x!h1F)qP@GQ3&88#X#l0vnQ39(3rBpHV+o0!ZC7HH3k4hoRtLtSjI%mi3X(h}5FyOR<ysc-^c3?necAo#Pwm6(<_BfX@1WO6OjDQCVK2)`at;G~>k!4<9c36)4gxE`obh;yLp*03sgPslq5gR$V@m>!rxPQ9Z$8go7PmPas*zRgEz2u3ZZ(FE#_@u1fRZupW1dT`0gL=!owDV>H!cV09`&*d=^+5W(s!J(1){`uSR#dmK{RBlXtm|{pZCy_?`Ay~QPiBFlc;GcZ^-ShC|+voq)w>9#e8gx|b=zQ_Yc=g9`-@f_hP&YbswyG91h<GRpg-St%_HyJ<mfvo@pkJXgC0ugH#xd`NDuGBt7X!exYz0I}!P3G#1+FujM|kDbYfyDw>ni{}yi}qzvlzAfVWCHO3Pz8-4(}`Sa0h1^+C7>nztKTo^)tpm(ZwYys4?Ns`JHLNGvQe_ixm57Gk)w(E$iSV5}AtV*ZtcgI$8%WD-?_Kl!88N8(HzSx=TvqRW6_-hc$Q|RhQ$V$>F5TCx<Y?fZxHZ@nQAbFDb8U{l;|Fm)i3p0pF1EaT^GKI&Q%-KQuvRW%&dE2^f3?Ry5*o+u}>>BRu0*3&O>GgFQKtQt*7FL0n(XS?6MQR=+)LLHjcvV$i=dwt!yi*EQ@vFC(;<*yYr_qc+@Cat(@bDEma0rC0TE|BN@C6j<a~7ok_jEofi?1p&t@s$Lx%h*vL~2w$L6iCGa}7}e;pS!oO2fRLawDe@Ne*q_EVM(^s~L`>tFE=^mvum)`|5|fcCO#%M8V)NQCNAgN_C*|bi#p~ArgJ>ikQPM1@f^P%h4vARYg2C&Ji>Xo8gK&&>VDbd{uC6gqN(6?~8d9QBXZJF~y9*XzSEMu>bV=p}?zdV1FG4Pg-mREgvGswf_|8nE@LBZxRI1B8@x9_jY|Q?X&DQmppL*Pn1Kq5*1DO^iH<awa#Ta}Ikl@r6Zam0g)4?l-6=Ktcc#M%HE8Zo`fb3?6j$o7BGH+oAU;57E^~B)eYv0`!_SkuAa}B>vPFCcU|An%bg2=v1(RC8rais{U2aF{EJL9a8!WeG&f`yiCRH9na-2y83?Z|ua^7pTvzX|{F{_UI7mv3Io;L?{2edG(}ng|GgeG)!@{d(rZ7C*4{jCk$=hdAB{14v3jXNXdJ1-d*|#j&8iniak30v^a0o&lw-mKc0vlzjK<-Aj58#~Abdhtn5tzdPOkCJ7V^MJt1vF_@8=FUk%&BGGYO-B!3iSitJc!1t(mHAQ>=lzSm>-Ygz1V`L)GFT6(#w1FVFQfeJP^4r}d=RqjsCPMh2goHrfeb`3y5Em8%mY^>W`w=M2v)Ig^ooj&0EB=Nm#{CzAwE~c36BR`|p96%0d7XU%qc5C~rsoZkG|BOdSe~ym0mwMUKmCL{BnaGUFbx*p+H9k-r8pU@lw&B?)^o;{&)rzj%&ol~&WHhGXaL<6<c|dYdFQft=rt;xtEhK%JhYY8V(MNbPYyd$Ab~(bMj*1xgRLhjfH4>TiM*fwrF<Ox<--?4fUa?B(@R%wqq=xa!YW{rq9}CB*3*RiY=a}^e*uR<&`0>)X^@1zL^|N-?%NdF8Ry+T`+ckHO-}o>P-RVONUON4oN|t_6v`a6UKDaIZyTGfW3llgtv2$08;`Sm%x}kdCWLGk8O1d|7O3Z=T7}U(4LQzO=j6Yl9$Kd-k(AVB66Za#mF8d0imn_NJ=_cTZ^$_Vht?j6rK2g!*3mu#l2y&$SEZ_KytYbJ-~7CfE_L`To-y?mVxI-dKB`LmZ;aQ`iMrK}|9xx*C~(j$@2qn$UqGa+1k%>l;TeYZb_dVyqrYgI-^J3dt?<1TdIftzSIsW;rSl)uqv{#k|7SJCi6X<d>I;)Lmz#iVV<?+^04ov0_)wUHd!erU{-TgV2}yXoj6aUc^=7;yx4Hg0>UwafGT&W*Pr$`^n{4l^qioV+%!D^PI{x}_^t;2u+RLb(#+t^vAYs|DnsQur%^Q%=XUovV5aWi%pB(%5g9Z*(0_&gW*LBw{f@*_M??buohVAuMB^xod^8yqE+P*+Lt{=l&zJOf{qNl?m+D7w)%xeBc4zeGJd5CdTg_FCXWj+Qz1oFFF*I3K`Xa0wm;mPxpZ(oM*o}YfJy$X=**^G%xbRx7G@=>r(VCjT-`sz<FL5fh^3xqVEZ+cW@JrjziWJA!>79|;N=VTF>iJN5KE{S7{GI4l$1s7TpJ3zaKBz`btnv?f0pP#-A-@FZfc=hJR+aK;)q%hN@pcI?p1*tmWSZx6hX2yR<GC4E;>+oy&bU&4rPlEV}#u;5iVY_YK&uR_POW5tLg<@T@j#L&FC3J@e6VpZTdoJ0SEIY)w-+hg(S2^fcp_01qvk6WSM_+%q1ZUj!O~hhUukmQ$tcf9*q#b)_N^I>egG2kJC>`21d>(|s=DTkw;m*Tp-;AMByJf>65Y}^=T&%0+4)<6HO)Jg6R~~MnD<@5}TzgqcfF=$d)#m7zt!-eaW~i-SLi0K@$U(*ALEzqi^|ont(&>3cuBqDNV8qwH6QHpk_p<}_`WgDne8YnyiRjq@z4Bdd6zUQi<nRq}{@!ApuA(j?UM9+lhXgd~myyi-BK+1w>J?Wf<Y5WVeX;1Cw)9W$$H0-A3p=HSMxC<eUK_l+!?Jr659w%Y3HEf=QRi3h_g<u;aP|~685q&(xrAGFkZf#nKvGI9l)FP)0hOA#ogRnGO3d6iZqe9Gjq^z|vh4;qJHj`P#5%{Wv^DKKHJHj1))mMpDY(+-Dc<<!iIl2&nYy6zsi%YmmEyBRc^uE>gf`8~IE1O!@F&R*b!wYNjuD-}#!7YA4$2ar{Z_)8?reuf(t@%T$*N=innxo$&2eWSYk#D@VNvdlw3mA*nlbchgZowXh(-wc?4A@%wkpV?QR}HetZkX<UM?!mWw-w!%;f<hn}Zw1_GPwv@rTaDcCm_f+B@%<K@6??*L(6XJm8+|HBfa<`92*QS=TM(_uM{S$0t6>ZNHPE9-s!b7^c(h-*$(0ph4eU*t(yUeKq)U9=d_Gjz=Zm7xX4kHx*H&Ug^9?Rr`QAoY)MWh)1RVD?Z(tOPcC3g-ZxA{XxKKl#W3})R7N;m%%mWtei1gGy+vL(F_?If9i$zcX0}QfZFWgdFvmP^ukfMmG{C?#+W3P`pYeexYIhcna%$PwF<bt'


import base64
import hashlib
import importlib.util
import zlib

def sha256(body):
    return hashlib.sha256(body).hexdigest()

sample_bytes = ''.join(json.dumps(r, ensure_ascii=False) + '\n' for r in SAMPLES).encode('utf-8')
crawler_bytes = zlib.decompress(base64.b85decode(CRAWLER_PAYLOAD))
if sha256(crawler_bytes) != CRAWLER_SHA256:
    raise ValueError('Snapshot crawler bị hỏng.')
if (len(SAMPLES) != EXPECTED_URL_COUNT or len({r['domain'] for r in SAMPLES}) != EXPECTED_DOMAIN_COUNT
        or len({r['url'] for r in SAMPLES}) != len(SAMPLES) or len({r['id'] for r in SAMPLES}) != len(SAMPLES)):
    raise ValueError('Danh sách mẫu thiếu hoặc trùng ID/URL.')
if (set(r['domain'] for r in SAMPLES) != set(TARGET_DOMAINS)
        or any(sum(r['domain'] == domain for r in SAMPLES) != SAMPLES_PER_DOMAIN for domain in TARGET_DOMAINS)):
    raise ValueError('Mẫu phải có đúng 20 URL cho mỗi domain VOV và Long Châu.')
manifest_path = RUN_DIR / 'manifest.json'
sample_path = RUN_DIR / 'sample.jsonl'
crawler_path = RUN_DIR / 'crawler_snapshot.py'
if manifest_path.exists():
    saved = json.loads(manifest_path.read_text(encoding='utf-8'))
    if (saved.get('probe_kind') != PROBE_KIND
            or saved.get('sample_sha256') != sha256(sample_bytes)
            or saved.get('source_sha256') != SOURCE_SHA256):
        raise ValueError('Run cũ không khớp mẫu của notebook này.')
    if not sample_path.exists() or sample_path.read_bytes() != sample_bytes:
        raise ValueError('Danh sách mẫu của run cũ đã thay đổi.')
    if not crawler_path.exists() or sha256(crawler_path.read_bytes()) != CRAWLER_SHA256:
        raise ValueError('Snapshot khác run cũ; hãy tạo run mới.')
    if json.loads((RUN_DIR / 'source_profile.json').read_text(encoding='utf-8')) != SOURCE_PROFILE:
        raise ValueError('Source profile của run cũ đã thay đổi.')
else:
    sample_path.write_bytes(sample_bytes)
    crawler_path.write_bytes(crawler_bytes)
    (RUN_DIR / 'source_profile.json').write_text(
        json.dumps(SOURCE_PROFILE, ensure_ascii=False, indent=2), encoding='utf-8')
    manifest_path.write_text(json.dumps({
        'created_at': datetime.now(timezone.utc).isoformat(),
        'probe_kind': PROBE_KIND, 'dataset': 'AIGuruTinix/ViBioMIR',
        'source': SOURCE_URL, 'source_file': SOURCE_FILE, 'source_revision': SOURCE_REVISION,
        'source_sha256': SOURCE_SHA256, 'source_rows': SOURCE_PROFILE['rows'],
        'sampling': SAMPLING_METHOD, 'seed': SAMPLE_SEED, 'selected': len(SAMPLES),
        'target_domains': list(TARGET_DOMAINS), 'samples_per_domain': SAMPLES_PER_DOMAIN,
        'selected_domains': EXPECTED_DOMAIN_COUNT,
        'baseline_archive': BASELINE_ARCHIVE, 'baseline_archive_sha256': BASELINE_ARCHIVE_SHA256,
        'sample_sha256': sha256(sample_bytes),
    }, ensure_ascii=False, indent=2), encoding='utf-8')

spec = importlib.util.spec_from_file_location('domain_probe_crawler', crawler_path)
crawler = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = crawler
spec.loader.exec_module(crawler)
priority_path = RUN_DIR / 'priority_probe_snapshot.py'
priority_bytes = zlib.decompress(base64.b85decode(PRIORITY_PAYLOAD))
if sha256(priority_bytes) != PRIORITY_SHA256:
    raise ValueError('Priority probe snapshot bị hỏng.')
if priority_path.exists() and sha256(priority_path.read_bytes()) != PRIORITY_SHA256:
    raise ValueError('Priority probe khác run cũ; tạo run mới.')
priority_path.write_bytes(priority_bytes)
priority_spec = importlib.util.spec_from_file_location('domain_priority_probe', priority_path)
priority_probe = importlib.util.module_from_spec(priority_spec)
sys.modules[priority_spec.name] = priority_probe
priority_spec.loader.exec_module(priority_probe)
if not isinstance(USER_AGENT, str) or not USER_AGENT.strip() or any(c in USER_AGENT for c in ('\r', '\n')):
    raise ValueError('USER_AGENT phải là chuỗi header hợp lệ.')
crawler.UA = USER_AGENT
crawler.ROBOT_AGENT = USER_AGENT.split('/', 1)[0]
(RUN_DIR / 'baseline_domains.json').write_text(json.dumps(BASELINE_DOMAINS, ensure_ascii=False, indent=2), encoding='utf-8')
print(f'Đã nạp {len(SAMPLES)} URL mẫu và crawler {crawler.VERSION}.')


## 4. Chuẩn bị bảng kết quả và phân loại lỗi

Output riêng: `vao_duoc.csv`, `khong_vao_duoc.csv`, `chua_chay.csv`,
`loi_theo_loai.csv`, `can_xem_noi_dung.csv`; bảng tổng là `ket_qua.csv`.
Các CSV dùng UTF-8 BOM để mở được tiếng Việt trong Excel.
`error_detail` giữ nguyên lỗi kỹ thuật; `reason` diễn giải tiếng Việt.
Mở `ket_qua.html` để lọc theo domain hoặc trạng thái.
`uu_tien_fix_domain.csv/html` xếp hostname còn lỗi theo số URL dataset giảm dần.
`tong_hop_domain.csv` gồm cả hostname đã phục hồi; `WEB_ERRORS.md` là báo cáo lỗi chi tiết.


In [ ]:
import csv
import html
import shutil
from collections import Counter
from IPython.display import HTML, FileLink, display

STATUS_REASONS = {
    'robots_unknown': 'Không xác định được chính sách robots.txt; crawler dừng trước khi tải trang.',
    'robots_denied': 'robots.txt trả HTTP 401/403; crawler dừng theo chính sách robots.',
    'robots_disallowed': 'robots.txt không cho phép crawler tải URL này.',
    'http_error': 'Trang đích trả HTTP ngoài khoảng 200–299.',
    'network_error': 'Không tải được trang do lỗi kết nối sau retry.',
    'unsafe_address': 'Địa chỉ đích không phải IP public được crawler cho phép.',
    'invalid_url': 'URL không hợp lệ hoặc dùng cổng ngoài 80/443.',
    'oversized': 'Phản hồi vượt giới hạn kích thước của crawler.',
    'response_deadline': 'Tải phản hồi vượt giới hạn thời gian tổng.',
    'rate_limited_deferred': 'Server yêu cầu chờ Retry-After quá lâu; crawler hoãn URL.',
    'redirect_limit': 'Vượt số lần redirect cho phép.',
    'unsupported_content_type': 'Đã tải phản hồi nhưng định dạng không được hỗ trợ.',
    'unexpected_error': 'Lỗi ngoài dự kiến; xem error_detail.',
}

def classify_error(status, detail, http_status):
    lower = detail.lower()
    if status == 'http_error':
        return f'page_http_{http_status}'
    if status == 'robots_disallowed':
        return 'robots_disallowed'
    if status.startswith('robots_') and 'robots http ' in lower:
        return 'robots_http_' + lower.split('robots http ', 1)[1].split()[0]
    if 'html instead of robots.txt' in lower:
        return 'robots_html_instead_of_txt'
    prefix = 'robots_' if status.startswith('robots_') else ''
    if 'waiting for domain lock' in lower:
        return prefix + 'domain_lock_budget'
    if 'request budget exhausted' in lower or 'retry budget exhausted' in lower:
        return prefix + 'budget_exhausted'
    if 'winerror 10013' in lower or 'network is unreachable' in lower:
        return prefix + 'environment_network_blocked'
    if any(s in lower for s in ('timed out', 'timeout')):
        return prefix + 'timeout'
    if any(s in lower for s in ('sslerror', 'certificate verify failed', 'ssl:')):
        return prefix + 'ssl_error'
    if any(s in lower for s in ('nameresolutionerror', 'getaddrinfo failed', 'name or service not known', 'failed to resolve')):
        return prefix + 'dns_error'
    if any(s in lower for s in ('connection refused', 'failed to establish a new connection', 'connection reset')):
        return prefix + 'connection_error'
    return status

def result_rows():
    rows = []
    for sample in sorted(SAMPLES, key=lambda s: (-SOURCE_PROFILE['host_counts'][s['domain']], s['domain'], s['sample_index'])):
        record_path = RUN_DIR / 'records' / f"{sample['id']}.json"
        record = json.loads(record_path.read_text(encoding='utf-8')) if record_path.exists() else {}
        if record and (record.get('id'), record.get('url')) != (sample['id'], sample['url']):
            raise ValueError('Checkpoint không khớp ID/URL nguồn.')
        status = record.get('status', 'pending')
        page_http = record.get('http_status')
        robots = record.get('robots', {})
        # Robots failure may happen before fetch() attaches its evidence to the record.
        if status.startswith('robots_') and not robots:
            from urllib.parse import urlsplit
            target = record.get('requests', [{}])[-1].get('url', sample['url']) if record.get('requests') else sample['url']
            if record.get('redirects'):
                target = record['redirects'][-1]['to']
            p = urlsplit(target)
            robot_file = RUN_DIR / 'robots' / (sha256(f'{p.scheme}://{p.netloc}'.encode()) + '.json')
            if robot_file.exists():
                robots = json.loads(robot_file.read_text(encoding='utf-8'))
        detail = record.get('error', '')
        browser_file = RUN_DIR / 'browser_records' / f"{sample['id']}.json"
        browser = json.loads(browser_file.read_text(encoding='utf-8')) if browser_file.exists() else {}
        if browser and (browser.get('id'), browser.get('url')) != (sample['id'], sample['url']):
            raise ValueError('Browser checkpoint không khớp ID/URL.')
        requests_access = ('CHUA_CHAY' if status == 'pending' else
                  'VAO_DUOC' if page_http is not None and 200 <= page_http < 300 else 'KHONG_VAO_DUOC')
        if DIRECT_PUBLIC_PROBE:
            access, effective_engine, effective = priority_probe.access_outcome(record, browser)
            if record.get('challenge_detected'):
                requests_access = 'KHONG_VAO_DUOC'
        else:
            access, effective_engine, effective = requests_access, 'requests', record
        flags = record.get('quality_flags', [])
        if status == 'pending':
            reason, error_type, stage = 'Chưa chạy URL này.', '', 'pending'
        elif status == 'extracted':
            stage = 'done'
            error_type = ('qc_' + record['quality_status'] if record.get('quality_status') in ('review', 'rejected') else '')
            reason = ('Tải được và nội dung đạt QC tự động.' if record.get('quality_status') == 'accepted'
                      else 'Tải được, nhưng nội dung cần xem lại hoặc bị QC loại: ' + ', '.join(flags))
        else:
            stage = ('robots' if status.startswith('robots_') else 'extraction' if access == 'VAO_DUOC' else 'fetch')
            error_type = record.get('error_kind') or classify_error(status, detail, page_http)
            reason = ('Trang trả HTML xác minh/JavaScript thay vì nội dung bài viết.'
                      if record.get('challenge_detected') else STATUS_REASONS.get(status, status))
        requests_reason = reason
        if effective_engine == 'selenium':
            stage = 'done_browser'
            reason = 'Selenium cải thiện tải/trích xuất so với Requests. QC: ' + effective.get('quality_status', 'unknown')
        rows.append({
            'domain': sample['domain'], 'dataset_url_count': SOURCE_PROFILE['host_counts'][sample['domain']],
            'sample_id': sample['id'], 'sample_url': sample['url'], 'access_result': access,
            'requests_access_result': requests_access, 'effective_engine': effective_engine,
            'effective_http_status': effective.get('http_status'),
            'effective_quality_status': effective.get('quality_status', ''),
            'effective_char_count': effective.get('char_count'),
            'effective_quality_flags': '; '.join(effective.get('quality_flags', [])),
            'http_status': page_http, 'robots_http_status': robots.get('status'),
            'robots_attempt_count': robots.get('attempt_count'),
            'robots_cache_hit': robots.get('cache_hit'),
            'robots_fetch_id': robots.get('fetch_id', ''),
            'robots_error_kind': robots.get('error_kind', ''),
            'robots_elapsed_seconds': robots.get('elapsed_seconds'),
            'robots_lock_wait_seconds': robots.get('lock_wait_seconds'),
            'crawl_status': status, 'failure_stage': stage, 'error_type': error_type,
            'reason': reason, 'requests_reason': requests_reason,
            'error_detail': detail, 'quality_status': record.get('quality_status', ''),
            'quality_flags': '; '.join(flags), 'char_count': record.get('char_count'),
            'title': record.get('title', ''), 'final_url': record.get('final_url', ''),
            'elapsed_seconds': record.get('elapsed_seconds'),
            'reused_from': record.get('reused_from', ''),
            'request_attempt_count': len(record.get('requests', [])),
            'retryable': record.get('retryable'), 'retry_deferred': record.get('retry_deferred', ''),
            'page_attempted': bool(record.get('requests')),
            'browser_status': browser.get('status', ''),
            'browser_http_status': browser.get('http_status'),
            'browser_quality_status': browser.get('quality_status', ''),
            'browser_char_count': browser.get('char_count'),
            'browser_final_url': browser.get('final_url', ''),
            'browser_error': browser.get('error', ''),
            'browser_attempt_count': browser.get('attempt_count'),
            'sample_role': sample['sample_role'], 'sample_index': sample['sample_index'],
        })
    return rows

def domain_rows(rows):
    result = []
    for domain in sorted(BASELINE_DOMAINS, key=lambda h: (-SOURCE_PROFILE['host_counts'][h], h)):
        selected = [r for r in rows if r['domain'] == domain]
        failed = [r for r in selected if r['access_result'] == 'KHONG_VAO_DUOC']
        content = [r for r in selected if r['access_result'] == 'VAO_DUOC' and r['effective_quality_status'] != 'accepted']
        pending = sum(r['access_result'] == 'CHUA_CHAY' for r in selected)
        errors = Counter(r['error_type'] for r in failed)
        errors.update('qc_' + (r['effective_quality_status'] or 'unknown') for r in content)
        result.append({
            'domain': domain, 'dataset_url_count': SOURCE_PROFILE['host_counts'][domain],
            'baseline_error_type': BASELINE_DOMAINS[domain]['error_type'],
            'selected_urls': len(selected), 'additional_urls': sum(r['sample_role'] == 'additional' for r in selected),
            'missing_additional_urls': BASELINE_DOMAINS[domain]['shortfall'],
            'completed_urls': len(selected) - pending, 'pending_urls': pending,
            'page_not_attempted': sum(not r['page_attempted'] and r['access_result'] != 'CHUA_CHAY' for r in selected),
            'accessible_urls': sum(r['access_result'] == 'VAO_DUOC' for r in selected),
            'access_failures': len(failed), 'content_issues': len(content),
            'total_problem_urls': len(failed) + len(content),
            'additional_problem_urls': sum(r['sample_role'] == 'additional' for r in failed + content),
            'error_types': '; '.join(f'{kind}: {n}' for kind, n in sorted(errors.items())),
            'test_status': 'CHUA_CHAY' if pending == len(selected) else 'CHUA_HOAN_TAT' if pending else 'CON_LOI' if failed or content else 'DA_PHUC_HOI',
        })
    return result

def write_csv(name, rows, fields):
    temporary = RUN_DIR / (name + '.tmp')
    with temporary.open('w', encoding='utf-8-sig', newline='') as stream:
        writer = csv.DictWriter(stream, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)
    temporary.replace(RUN_DIR / name)

def save_outputs(show=True):
    rows = result_rows()
    fields = list(rows[0])
    counts = Counter(r['access_result'] for r in rows)
    summary = {'selected': len(rows), 'vao_duoc': counts['VAO_DUOC'],
               'khong_vao_duoc': counts['KHONG_VAO_DUOC'], 'chua_chay': counts['CHUA_CHAY'],
               'crawl_status_counts': dict(Counter(r['crawl_status'] for r in rows)),
               'quality_status_counts': dict(Counter(r['quality_status'] for r in rows if r['quality_status'])),
               'effective_quality_status_counts': dict(Counter(r['effective_quality_status'] for r in rows if r['effective_quality_status'])),
               'created_at': datetime.now(timezone.utc).isoformat()}
    domains = domain_rows(rows)
    priority = [dict(priority_rank=i, **r) for i, r in enumerate((r for r in domains if r['total_problem_urls']), 1)]
    summary.update(hostnames=len(domains), domains_with_errors=len(priority),
                   priority_order='dataset_url_count descending', user_agent=USER_AGENT,
                   page_not_attempted=sum(not r['page_attempted'] and r['access_result'] != 'CHUA_CHAY' for r in rows),
                   additional_urls=sum(r['sample_role'] == 'additional' for r in rows))
    summary.update(request_mode='direct_public_page_probe' if DIRECT_PUBLIC_PROBE else 'robots_enforced',
                   requests_http_2xx=sum(r['requests_access_result'] == 'VAO_DUOC' for r in rows),
                   browser_recovered=sum(r['access_result'] == 'VAO_DUOC' and r['effective_engine'] == 'selenium' for r in rows),
                   browser_access_recovered=sum(r['access_result'] == 'VAO_DUOC' and r['effective_engine'] == 'selenium' and r['requests_access_result'] != 'VAO_DUOC' for r in rows),
                   browser_content_improved=sum(r['access_result'] == 'VAO_DUOC' and r['effective_engine'] == 'selenium' and r['requests_access_result'] == 'VAO_DUOC' for r in rows),
                   browser_tested=sum(bool(r['browser_status']) for r in rows),
                   browser_http_2xx=sum(r['browser_http_status'] is not None and 200 <= r['browser_http_status'] < 300 for r in rows))
    write_csv('tong_hop_domain.csv', domains, list(domains[0]))
    write_csv('uu_tien_fix_domain.csv', priority, ['priority_rank', *domains[0]])
    (RUN_DIR / 'uu_tien_fix_domain.json').write_text(json.dumps(priority, ensure_ascii=False, indent=2), encoding='utf-8')
    write_csv('ket_qua.csv', rows, fields)
    write_csv('requests_vs_selenium.csv', rows, fields)
    for value, filename in [('VAO_DUOC', 'vao_duoc.csv'), ('KHONG_VAO_DUOC', 'khong_vao_duoc.csv'), ('CHUA_CHAY', 'chua_chay.csv')]:
        write_csv(filename, [r for r in rows if r['access_result'] == value], fields)
    write_csv('can_xem_noi_dung.csv', [r for r in rows if r['access_result'] == 'VAO_DUOC' and r['effective_quality_status'] != 'accepted'], fields)
    failures = Counter(r['error_type'] for r in rows if r['access_result'] == 'KHONG_VAO_DUOC')
    errors = [{'error_type': kind, 'domain_count': len({r['domain'] for r in rows if r['access_result'] == 'KHONG_VAO_DUOC' and r['error_type'] == kind}),
               'url_count': n,
               'domains': '; '.join(sorted({r['domain'] for r in rows if r['access_result'] == 'KHONG_VAO_DUOC' and r['error_type'] == kind}))}
              for kind, n in failures.most_common()]
    write_csv('loi_theo_loai.csv', errors, ['error_type', 'domain_count', 'url_count', 'domains'])
    (RUN_DIR / 'ket_qua.json').write_text(json.dumps(rows, ensure_ascii=False, indent=2), encoding='utf-8')
    (RUN_DIR / 'summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
    md = ['# Các website còn lỗi sau lượt test bổ sung', '',
          f"Thời điểm UTC: {summary['created_at']}. Run: `{RUN_DIR.name}`.", '',
          f"Đã chạy {len(rows)-counts['CHUA_CHAY']}/{len(rows)} URL thuộc {len(domains)} hostname. "
          f"Tải được sau Requests/Selenium: {counts['VAO_DUOC']}; chưa tải được: {counts['KHONG_VAO_DUOC']}; chưa chạy: {counts['CHUA_CHAY']}.", '',
          f"Requests tải được: {summary['requests_http_2xx']}; chọn kết quả Selenium: {summary['browser_recovered']} "
          f"(phục hồi truy cập: {summary['browser_access_recovered']}; cải thiện nội dung: {summary['browser_content_improved']}). "
          'Các cột lỗi Requests vẫn giữ nguyên dù Selenium phục hồi thành công.', '',
          f"{summary['page_not_attempted']} URL đã xử lý nhưng chưa request bài viết vì dừng trước bước tải bài. "
          'Các trường hợp này được phân biệt với bài viết thực sự trả HTTP lỗi.', '',
          f'HTTP User-Agent: `{USER_AGENT}`.', '',
          f"Bộ mẫu có {len(rows)} URL thuộc {len(domains)} hostname, gồm {summary['additional_urls']} URL bổ sung. "
          'Gồm lỗi truy cập và lỗi nội dung/QC. Số URL dataset là số dòng của hostname, chưa loại URL trùng.', '',
          'Ưu tiên theo số URL dataset giảm dần; không suy ra toàn bộ URL của domain đều bị lỗi. '
          'Dừng ở robots là chưa request bài viết, không đồng nghĩa website không mở được trong trình duyệt.', '',
          '| Ưu tiên | Hostname | Số URL dataset | Mẫu đã chạy | Lỗi truy cập | Lỗi nội dung | Loại lỗi |',
          '|---:|---|---:|---:|---:|---:|---|']
    for item in priority:
        md.append(f"| {item['priority_rank']} | {item['domain']} | {item['dataset_url_count']:,} | {item['completed_urls']}/{item['selected_urls']} | {item['access_failures']} | {item['content_issues']} | {item['error_types']} |")
    md += ['', '## Domain đã phục hồi trong các mẫu', '']
    md += [f"- {r['domain']}: {r['accessible_urls']}/{r['selected_urls']} URL tải được và đạt QC." for r in domains if r['test_status'] == 'DA_PHUC_HOI'] or ['Chưa xác định domain phục hồi hoàn toàn.']
    md += ['', '## Domain thiếu URL bổ sung', '']
    md += [f"- {r['domain']}: lấy được {r['additional_urls']}/4 URL mới; dataset có {r['dataset_url_count']} dòng." for r in domains if r['missing_additional_urls']]
    md += ['', '## Chi tiết lỗi theo domain', '']
    for item in priority:
        md += [f"### {item['priority_rank']}. {item['domain']}", '',
               f"{item['dataset_url_count']:,} URL nguồn; {item['page_not_attempted']} mẫu đã xử lý nhưng chưa request bài viết. "
               f"Lỗi baseline: `{item['baseline_error_type']}`.", '']
        for row in rows:
            if row['domain'] != item['domain'] or row['access_result'] == 'CHUA_CHAY' or (row['access_result'] == 'VAO_DUOC' and row['effective_quality_status'] == 'accepted'):
                continue
            detail = (row['error_detail'] or row['quality_flags']).replace('\n', ' ').replace('`', "'")
            md += [f"- ID {row['sample_id']} ({row['sample_role']}): [URL](<{row['sample_url']}>) — `{row['error_type']}`. "
                   f"HTTP trang: {row['http_status']}; HTTP robots: {row['robots_http_status']}. {row['reason']} Lỗi gốc: `{detail}`."]
        md.append('')
    (RUN_DIR / 'WEB_ERRORS.md').write_text('\n'.join(md), encoding='utf-8')

    visible_fields = ['domain', 'dataset_url_count', 'sample_id', 'sample_url', 'sample_role', 'access_result',
                      'http_status', 'robots_http_status', 'robots_attempt_count', 'robots_cache_hit', 'error_type', 'reason', 'error_detail',
                      'quality_status', 'quality_flags', 'char_count']
    labels = ['Domain', 'Số URL nguồn', 'ID mẫu', 'URL mẫu', 'Mẫu cũ/mới', 'Vào được?', 'HTTP trang',
              'HTTP robots', 'Lần thử robots', 'Cache hit robots', 'Loại lỗi', 'Lý do', 'Lỗi gốc', 'QC', 'Cờ QC', 'Số ký tự']
    if DIRECT_PUBLIC_PROBE:
        visible_fields = ['domain', 'dataset_url_count', 'sample_id', 'sample_url', 'http_status',
                          'error_type', 'error_detail', 'quality_status', 'char_count',
                          'browser_status', 'browser_http_status', 'browser_quality_status', 'browser_char_count', 'browser_error',
                          'access_result', 'effective_engine']
        labels = ['Domain', 'Số URL nguồn', 'ID', 'URL', 'HTTP Requests', 'Lỗi Requests', 'Chi tiết',
                  'QC Requests', 'Ký tự Requests', 'Selenium', 'HTTP Selenium', 'QC Selenium', 'Ký tự Selenium', 'Lỗi Selenium',
                  'Kết quả cuối', 'Engine tải được']
    def esc(value):
        return html.escape(str(value if value is not None else ''), quote=True)
    body = []
    for row in rows:
        cells = []
        for field in visible_fields:
            value = row[field]
            text = (f'<a href="{esc(value)}" target="_blank" rel="noopener noreferrer">{esc(value)}</a>'
                    if field == 'sample_url' else esc(value))
            cells.append('<td>' + text + '</td>')
        body.append(f'<tr data-access="{row["access_result"]}">' + ''.join(cells) + '</tr>')
    style = """<style>body{font:14px/1.5 system-ui,sans-serif;margin:20px;color:#172b4d}
    .table-wrap{overflow:auto;max-height:70vh;border:1px solid #ccd4df}table{border-collapse:collapse;width:100%}
    th,td{padding:9px;border-bottom:1px solid #dce2e9;text-align:left;vertical-align:top}th{background:#e8eef6;position:sticky;top:0}
    td{min-width:95px;max-width:330px;overflow-wrap:anywhere}td:nth-child(4),td:nth-child(9),td:nth-child(10){min-width:260px}
    input,select{padding:8px;margin:0 10px 14px 0}a{color:#1654a0}</style>"""
    table = '<div class="table-wrap"><table><thead><tr>' + ''.join('<th>' + esc(label) + '</th>' for label in labels)
    table += '</tr></thead><tbody>' + ''.join(body) + '</tbody></table></div>'
    heading = f'<h1>Kết quả thử crawl {len(rows)} URL / {len(domains)} hostname</h1><p>Vào được: {counts["VAO_DUOC"]} · Không vào được: {counts["KHONG_VAO_DUOC"]} · Chưa chạy: {counts["CHUA_CHAY"]}</p>'
    heading += '<p>Vào được tính cả Requests và Selenium phục hồi; trang challenge được loại. Các cột Requests vẫn giữ lỗi gốc, QC là kết quả riêng. Kết quả chỉ áp dụng cho các URL mẫu.</p>'
    controls = '<input id="search" placeholder="Tìm domain, URL hoặc lỗi" aria-label="Tìm kiếm"><select id="access" aria-label="Kết quả"><option value="">Tất cả</option><option>VAO_DUOC</option><option>KHONG_VAO_DUOC</option><option>CHUA_CHAY</option></select><span id="count"></span>'
    priority_fields = ['priority_rank', 'domain', 'dataset_url_count', 'selected_urls', 'completed_urls', 'pending_urls', 'access_failures', 'content_issues', 'error_types']
    priority_labels = ['Ưu tiên', 'Domain', 'Số URL dataset', 'Mẫu', 'Đã chạy', 'Chưa chạy', 'Lỗi truy cập', 'Lỗi nội dung', 'Loại lỗi']
    priority_heading = '<h2>Domain còn lỗi — số URL dataset giảm dần</h2><p>Gồm lỗi truy cập và lỗi nội dung/QC. Số URL nguồn không phải số URL đã xác minh lỗi.</p>'
    priority_table = '<div class="table-wrap"><table id="priority"><thead><tr>' + ''.join('<th>'+esc(label)+'</th>' for label in priority_labels)
    priority_table += '</tr></thead><tbody>' + ''.join('<tr>' + ''.join('<td>'+esc(r[f])+'</td>' for f in priority_fields) + '</tr>' for r in priority) + '</tbody></table></div>'
    (RUN_DIR / 'uu_tien_fix_domain.html').write_text('<!doctype html><html lang="vi"><meta charset="utf-8"><title>Ưu tiên domain lỗi</title>'+style+priority_heading+priority_table+'</html>', encoding='utf-8')
    javascript = """<script>const rs=[...document.querySelectorAll('tbody tr')],s=document.getElementById('search'),a=document.getElementById('access');
    function filter(){let n=0;for(const r of rs){const ok=r.textContent.toLowerCase().includes(s.value.toLowerCase())&&(!a.value||r.dataset.access===a.value);r.hidden=!ok;n+=ok?1:0}document.getElementById('count').textContent=n+' hostname'}
    s.addEventListener('input',filter);a.addEventListener('change',filter);filter();</script>"""
    javascript = javascript.replace("document.querySelectorAll('tbody tr')", "document.querySelectorAll('tr[data-access]')").replace("n+' hostname'", "n+' URL'")
    (RUN_DIR / 'ket_qua.html').write_text('<!doctype html><html lang="vi"><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>Kết quả crawl domain</title>' + style + heading + priority_heading + priority_table + '<h2>Từng URL mẫu</h2>' + controls + table + javascript + '</html>', encoding='utf-8')
    # Archive outside the run directory to avoid including the archive itself.
    zip_path = Path(shutil.make_archive(str(RUN_DIR), 'zip', root_dir=RUN_DIR.parent, base_dir=RUN_DIR.name))
    if show:
        print(json.dumps(summary, ensure_ascii=False, indent=2))
        display(HTML(style + priority_heading + priority_table))
        display(HTML(style + heading + table))
        print('File kết quả:', RUN_DIR)
        print('ZIP:', zip_path)
    return rows, summary, zip_path

# Create reports before networking, so partial runs have explicit CHUA_CHAY rows.
rows, summary, zip_path = save_outputs(show=False)
print('Đã chuẩn bị báo cáo và phân loại lỗi.')


## 5. Tải trang và đối chiếu Selenium

Chạy Requests trước, từng URL tuần tự trong một domain; các domain chạy song song.
Mỗi URL có timeout connect/read và ngân sách riêng, không tiêu hao ngân sách vì các mẫu cùng domain chờ nhau.
Ngân sách được kiểm tra giữa các bước; DNS và thử nhiều IP có thể làm thời gian thực vượt mức đặt.
Theo `BROWSER_TEST_MODE`, Selenium thử lại URL Requests lỗi hoặc HTTP 200 nhưng text rỗng/QC lỗi;
có thể đặt `off` để chỉ chạy Requests. Nếu bật Selenium, Chrome được kiểm tra trước lượt Requests.
Trên Linux, notebook tự cài thư viện Chrome còn thiếu bằng apt (gồm ATK); Selenium Manager tìm/tải
Chrome và driver. Preflight thất bại sẽ dừng ô crawl, lưu `browser_setup_error.json`,
`browser_runtime.json`, `browser_preflight.json` và log Chrome để chẩn đoán.
Requests và Selenium có kết quả riêng; DOM trống/challenge không được ghi thành tải được bài.
Requests retry tối đa 3 lần; lỗi mạng được đóng pool kết nối cũ trước lần thử tiếp theo.
Selenium dùng User-Agent thực và retry timeout tối đa 2 lần; 403/challenge không được retry liên tục.
`access_result` tính cả trường hợp Selenium phục hồi, còn các cột lỗi Requests vẫn giữ nguyên.
Kết quả cuối chọn engine có QC tốt hơn; kết quả tương đương giữ Requests.
Parser 120ask lấy câu hỏi/câu trả lời; cnkang lấy thân bài, bỏ menu/bài liên quan.
`summary.json` tách số URL phục hồi truy cập và số URL cải thiện nội dung nhờ Selenium.
Để thử lại riêng nhóm lỗi, đặt `RECOVERY_RUN_DIR` là đường dẫn run cũ và giữ `RESUME_RUN_DIR = None`.
Recovery tạo run mới, kiểm tra hash raw 2xx trước khi reuse, để các checkpoint lỗi được tải lại.
Để chạy lại toàn bộ bằng cấu hình mới, giữ `RESUME_RUN_DIR = None` và Run All.

Mặc định `failed_only` tự chuyển mọi lỗi Requests/trích xuất, text rỗng và QC `review`/`rejected`
sang Selenium, kể cả PDF cần kiểm tra lại. `off` chủ động tắt đối chiếu Selenium;
robots bị chặn và URL không hợp lệ/địa chỉ không được phép vẫn được giữ nguyên.

Tiến độ và cảnh báo QC (review, domain_parser_fallback, …) được tổng hợp mỗi LOG_EVERY URL.
Lỗi HTTP/mạng, trang challenge và nội dung rỗng vẫn in ngay kèm URL và nguyên nhân.


In [ ]:
import argparse
import os
import socket
import time

# DNS check is diagnostic only; failure here does not stop the crawl.
network_path = RUN_DIR / 'network_check.json'
if not network_path.exists():
    network = {
        'checked_at': datetime.now(timezone.utc).isoformat(),
        'environment': 'kaggle' if Path('/kaggle/working').is_dir() else 'colab' if Path('/content').is_dir() else 'local',
        'python': sys.version, 'platform': sys.platform,
        'proxy_environment_present': {key: bool(os.environ.get(key)) for key in
                                      ('HTTP_PROXY', 'HTTPS_PROXY', 'ALL_PROXY', 'NO_PROXY')},
        'dns': [],
    }
    for domain in NETWORK_CHECK_DOMAINS:
        start = time.monotonic()
        entry = {'domain': domain}
        try:
            entry['addresses'] = sorted({r[4][0] for r in socket.getaddrinfo(domain, 443)})
        except OSError as exc:
            entry.update(exception_class=type(exc).__name__, error=str(exc))
        entry['elapsed_seconds'] = round(time.monotonic() - start, 3)
        network['dns'].append(entry)
    network_path.write_text(json.dumps(network, ensure_ascii=False, indent=2), encoding='utf-8')
print('Network check:', network_path)

try:
    if DIRECT_PUBLIC_PROBE:
        if BROWSER_TEST_MODE != 'off':
            priority_probe.browser_preflight(crawler, RUN_DIR, user_agent=BROWSER_USER_AGENT,
                                             chrome_binary=CHROME_BINARY, load_timeout=BROWSER_LOAD_TIMEOUT)
        if RECOVERY_RUN_DIR:
            recovery = priority_probe.seed_successful_requests(crawler, RUN_DIR, SAMPLES, RECOVERY_RUN_DIR)
            print('Recovery:', len(recovery['reused_ids']), 'URL reuse raw;', len(recovery['retry_ids']), 'URL thử lại.')
        records = priority_probe.crawl_requests(
            crawler, RUN_DIR, SAMPLES, workers=WORKERS, delay=DELAY,
            attempts=PAGE_ATTEMPTS, user_agent=USER_AGENT,
            timeout=(PAGE_CONNECT_TIMEOUT, PAGE_READ_TIMEOUT), budget=PAGE_REQUEST_BUDGET, max_new=MAX_NEW, log_every=LOG_EVERY)
        browser_samples = priority_probe.select_browser_samples(SAMPLES, records, BROWSER_TEST_MODE)
        if browser_samples:
            priority_probe.crawl_browser(crawler, RUN_DIR, browser_samples,
                                        user_agent=BROWSER_USER_AGENT, chrome_binary=CHROME_BINARY, delay=DELAY,
                                        load_timeout=BROWSER_LOAD_TIMEOUT, attempts=BROWSER_ATTEMPTS, log_every=LOG_EVERY)
        crawler.report(argparse.Namespace(out=str(RUN_DIR)))
    else:
        crawler.locked_crawl(argparse.Namespace(
            out=str(RUN_DIR), workers=WORKERS, delay=DELAY, attempts=ATTEMPTS, max_new=MAX_NEW,
            log_every=LOG_EVERY,
            robots_attempts=ROBOTS_ATTEMPTS, robots_connect_timeout=ROBOTS_CONNECT_TIMEOUT,
            robots_read_timeout=ROBOTS_READ_TIMEOUT, robots_retry_budget=ROBOTS_RETRY_BUDGET,
            robots_failure_ttl=ROBOTS_FAILURE_TTL,
        ))
finally:
    rows, summary, zip_path = save_outputs()


## 6. Xem lỗi riêng và tải output

`khong_vao_duoc.csv` chứa domain, URL, loại lỗi, lý do tiếng Việt và lỗi gốc.
`loi_theo_loai.csv` gom số hostname và số URL theo từng lỗi (HTTP 403, timeout,
robots trả HTML, robots cấm crawl, DNS, SSL…). File JSON trong `records/`
giữ trace request/redirect; `robots/` giữ bằng chứng robots.txt.
`network_check.json` ghi DNS của mọi hostname trong bộ mẫu để đối chiếu Kaggle/local;
`priority_probe_config.json` ghi User-Agent/header, timeout và chế độ tải trang.
`requests_vs_selenium.csv` và `browser_records/` giữ kết quả trình duyệt riêng;
`browser_setup_error.json` được ghi nếu Chrome/driver không khởi động được.
`browser_runtime.json`, `browser_preflight.json`, `chrome_dependencies.log` và `chromedriver.log`
ghi kiểm tra môi trường/cài thư viện/khởi động Chrome (log cài apt chỉ có khi cần cài thêm).
ZIP gồm kết quả, raw, ảnh chụp và snapshot để kiểm tra riêng.

Kaggle: tải ZIP trong **Output**. Colab: ô dưới hiện nút tải ZIP.
Jupyter local: dùng các liên kết file hoặc mở đường dẫn output được in ra.


In [ ]:
# Có thể chạy lại ô này để xuất báo cáo từ checkpoint mà không crawl thêm.
rows, summary, zip_path = save_outputs(show=False)
failed_rows = [r for r in rows if r['access_result'] == 'KHONG_VAO_DUOC']
print(f'Không vào được: {len(failed_rows)}/{len(rows)} URL, thuộc {len({r["domain"] for r in failed_rows})} hostname')
for row in failed_rows:
    print(f"{row['domain']} | {row['error_type']} | {row['error_detail']}")
print('\nOutput:', RUN_DIR)
for name in ['WEB_ERRORS.md', 'uu_tien_fix_domain.html', 'uu_tien_fix_domain.csv', 'tong_hop_domain.csv',
             'requests_vs_selenium.csv',
             'ket_qua.html', 'ket_qua.csv', 'vao_duoc.csv', 'khong_vao_duoc.csv',
             'loi_theo_loai.csv', 'can_xem_noi_dung.csv', 'chua_chay.csv']:
    display(FileLink(str(RUN_DIR / name)))
display(FileLink(str(zip_path)))

if 'google.colab' in sys.modules:
    import ipywidgets as widgets
    from google.colab import files
    button = widgets.Button(description='Tải ZIP kết quả')
    button.on_click(lambda _: files.download(str(zip_path)))
    display(button)
